# 12 · Tumour-only, lungs-only, habitus and harmonized inputs (plan 7o, G1–G3)

**Project:** Acquisition shortcuts in public cancer imaging (Hamza El Massaoudy)
**Notebook written:** 7 Nov 2026 · **Needs:** `results/01_inventory/` to `results/11_external_lung/` · **Plan:** section
**7o** (`protocol/analysis_plan_7o.md`), written on 7 Nov 2026 before any image, mask or embedding of this notebook existed.

**What this notebook does, in plain words**

All earlier probes looked at a whole CT volume. This notebook asks where the lung-histology signal sits:
- **G1 · tumour only and lungs only:** everything outside the automatically segmented tumour (or the lungs) is replaced
  by air, and the same CT-FM and Merlin probes are fitted and transferred from TCGA lung to the four lung cohorts;
- **G2 · body habitus and positioning:** seven measures per scan (body shape, subcutaneous fat, arms, table height, body
  centring, scan length, lung volume), by class, as a model of their own, and the background-only probe after removing
  them;
- **G3 · harmonized geometry:** every scan resampled to 1 × 1 × 5 mm and cropped to a fixed field around the lungs, then
  probed again.
It also saves small harmonized volumes for notebook 13 (an end-to-end 3D network, G4).

**Scans:** exactly the 985 lung scans already embedded in notebooks 05, 07, 08 and 11 (TCGA lung 104, CPTAC lung 73,
NSCLC Radiogenomics 199, NSCLC-Radiomics 203, Lung-PET-CT-Dx 312) and the 94 LIDC-IDRI scans of primary versus metastatic
cancer. Each is downloaded again; the rebuilt body mask must match the saved one.

**Segmentation:** TotalSegmentator 2.10.0 runs in its own small Python environment, so the probe code keeps the package
versions of notebooks 05–11. Its `lung_nodules` model is checked against the manual tumour contours (GTV-1) of
NSCLC-Radiomics.

**Rehearsal first.** With `RUN_MODE = "both"`, the analyses run first on **shuffled labels** and pass an automatic gate
before the real labels are used.

**How to run:** in Colab choose *Runtime → Change runtime type → A100 GPU (or L4), High-RAM*, then *Runtime → Run all*.
Expect **10–16 hours** in total (about 1,080 scans to download, segment twice and embed); Colab may stop before that.
**If Colab disconnects, run all again:** every finished scan is kept on Drive and is not redone. Outputs go to
`results/12_image_experiments/`. The last cell stops the runtime.

In [ ]:
# 1. Setup: Drive, folders, GPU (notebook 11's setup; the outputs of this notebook go to 12_image_experiments)
import os, re, sys, json, time, copy, shutil, zipfile, hashlib, datetime, platform, warnings, subprocess
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RUN_MODE = 'both'  # @param ["both", "rehearsal", "real"]

if IN_COLAB:
    PROJECT = Path('/content/drive/MyDrive/Acquisition-Shortcut-Audit')
    WORK = Path('/content/asa12_work')          # Colab's own disk: DICOM, NIfTI and masks, never Drive
else:  # local code tests only
    PROJECT = Path(os.environ['ASA_PROJECT_DIR'])
    WORK = Path(os.environ['ASA_WORK_DIR'])
    RUN_MODE = os.environ.get('ASA_TEST_RUN_MODE', 'rehearsal')
assert PROJECT.exists(), f'Project folder not found: {PROJECT}. Is Colab signed in to the project Drive?'
assert RUN_MODE in ('both', 'rehearsal', 'real'), RUN_MODE

IN01 = PROJECT / 'results' / '01_inventory'
IN02 = PROJECT / 'results' / '02_tasks_features'
IN03 = PROJECT / 'results' / '03_audit'
IN04 = PROJECT / 'results' / '04_posthoc'
IN05 = PROJECT / 'results' / '05_image_check'
IN06 = PROJECT / 'results' / '06_image_posthoc'
IN07 = PROJECT / 'results' / '07_screen_validation'
IN08 = PROJECT / 'results' / '08_transfer'
IN09 = PROJECT / 'results' / '09_transfer_posthoc'
IN11 = PROJECT / 'results' / '11_external_lung'
CACHE05, CACHE06, CACHE07, CACHE08 = IN05 / 'cache', IN06 / 'cache', IN07 / 'cache', IN08 / 'cache'   # read only
CACHE = IN11 / 'cache'                           # notebook 11's cache, read only (TCIA replies, headers)
LABEL_DIR = IN11 / 'labels'                      # notebook 11's pinned Lung1 label file, read only
OUT_ROOT = PROJECT / 'results' / '12_image_experiments'
CACHE12 = OUT_ROOT / 'cache'                     # label-free outputs of this notebook
for d in (IN01, IN02, IN03, IN04):
    assert (d / 'run_manifest.json').exists(), f'{d} is missing: run the earlier notebooks first.'
for d in (IN05, IN06, IN07, IN08, IN09, IN11):
    for m in ('rehearsal', 'real'):
        assert (d / m / 'run_manifest.json').exists(), f'{d / m} is missing: run notebooks 05 to 11 first.'
for d in (CACHE12 / 'records', CACHE12 / 'embeddings', CACHE12 / 'cnn', CACHE12 / 'qc', CACHE12 / 'ts',
          WORK / 'nifti', WORK / 'dicom', WORK / 'seg'):
    d.mkdir(parents=True, exist_ok=True)

RUN_STARTED = datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds')
free_gb = shutil.disk_usage(WORK).free / 1e9
print('Project folder:', PROJECT)
print('Outputs to    :', OUT_ROOT)
print('Work disk     :', WORK, f'({free_gb:.0f} GB free)')
print('Run mode      :', RUN_MODE)
print('Run started   :', RUN_STARTED, '(UTC)')
if free_gb < 60:
    print('WARNING: less than 60 GB free on the work disk. Use an A100 or L4 runtime (they have more disk).')

In [ ]:
# 2. Packages (exact versions; installed into Colab only) (notebook 11, unchanged)
if IN_COLAB:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'monai==1.6.1', 'SimpleITK==2.5.6',
                    'pydicom==3.0.2', 'nibabel==5.4.2', 'merlin-vlm==0.0.7'], check=True)

import numpy as np
import pandas as pd
import requests
import scipy
from scipy import ndimage, sparse
from scipy.stats import rankdata, spearmanr
import sklearn
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, KFold, GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
import joblib
from joblib import Parallel, delayed
import pydicom
import SimpleITK as sitk
import nibabel
import torch
import torchvision
import monai
from monai.transforms import (Compose, LoadImage, EnsureType, Orientation, Spacing, ScaleIntensityRange, SpatialCrop)
from monai.transforms.utils import generate_spatial_bounding_box
from monai.inferers import SlidingWindowSplitter
from monai.networks.nets.segresnet_ds import SegResEncoder
import huggingface_hub
from huggingface_hub import hf_hub_download
import safetensors.torch
import merlin
from merlin.models import i3res
from merlin.data.monai_transforms import ImageTransforms as MERLIN_TRANSFORMS
import importlib.metadata

warnings.filterwarnings('ignore', message='.*pixdim.*')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 60)
N_JOBS = -1
N_CPU = os.cpu_count()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

def show(df, n=30):
    """Display a table in Colab, or print it in a plain Python run."""
    try:
        display(df.head(n))
    except NameError:
        print(df.head(n).to_string())

def md5(p):
    h = hashlib.md5()
    with open(p, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

def write_json_atomic(path, obj):
    tmp = path.with_name(path.name + '.tmp')
    tmp.write_text(json.dumps(obj, default=_jsonable))
    tmp.replace(path)

def _jsonable(o):
    return o.item() if hasattr(o, 'item') else str(o)

VERSIONS = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
            'scipy': scipy.__version__, 'scikit-learn': sklearn.__version__, 'joblib': joblib.__version__,
            'torch': torch.__version__, 'torchvision': torchvision.__version__, 'monai': monai.__version__,
            'SimpleITK': sitk.Version.VersionString(), 'pydicom': pydicom.__version__,
            'nibabel': nibabel.__version__, 'huggingface_hub': huggingface_hub.__version__,
            'merlin-vlm': importlib.metadata.version('merlin-vlm'), 'cpus': N_CPU,
            'gpu': torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'none'}
print(VERSIONS)
if IN_COLAB and DEVICE != 'cuda':
    print('WARNING: no GPU. Choose Runtime → Change runtime type → L4 GPU, then run all again.')

In [ ]:
# 2b. TotalSegmentator in its own Python 3.12 environment (so that numpy, scikit-learn etc. stay as in notebooks
# 05-11, and independent of Colab's Python version: TotalSegmentator 2.10 needs numpy < 2, which has no build for 3.13)
TS_VERSION = '2.10.0'
TS_TORCH = '2.5.1'                     # pinned, so that the segmentation environment does not drift
TS_NNUNET = '2.6.2'                    # newer nnU-Net no longer accepts TotalSegmentator 2.10's custom trainer
TS_PYTHON, UV_VERSION = '3.12', '0.4.30'
TS_HOME = (Path('/content') if IN_COLAB else WORK) / 'ts_env'
TS_WEIGHTS = (Path('/content') if IN_COLAB else WORK) / 'ts_weights'      # model weights, downloaded once per machine
_bin = TS_HOME / ('Scripts' if os.name == 'nt' else 'bin')
TS_BIN = _bin / ('TotalSegmentator.exe' if os.name == 'nt' else 'TotalSegmentator')
TS_PY = _bin / ('python.exe' if os.name == 'nt' else 'python')
FAKE = os.environ.get('ASA_TEST_FAKE_IMAGES') == '1' and not IN_COLAB
if not TS_BIN.exists() and not FAKE:
    shutil.rmtree(TS_HOME, ignore_errors=True)                      # a half-made environment from an earlier attempt
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', f'uv=={UV_VERSION}'], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'venv', '--python', TS_PYTHON, str(TS_HOME)], check=True)
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(TS_PY), f'TotalSegmentator=={TS_VERSION}',
                    f'torch=={TS_TORCH}', f'nnunetv2=={TS_NNUNET}'], check=True)
TS_ENVIRON = {**os.environ, 'TOTALSEG_HOME_DIR': str(TS_WEIGHTS), 'PYTHONIOENCODING': 'utf-8', 'PYTHONUTF8': '1'}
TS_WEIGHTS.mkdir(parents=True, exist_ok=True)
TS_CHECK = None
if not FAKE:                                                         # check the environment before any long work
    r = subprocess.run([str(TS_PY), '-c', 'import sys, torch, importlib.metadata as m; print(sys.version.split()[0], '
                        'torch.__version__, torch.cuda.is_available(), m.version("TotalSegmentator"), m.version("nnunetv2"), '
                        'm.version("numpy"))'], capture_output=True, text=True, env=TS_ENVIRON)
    assert r.returncode == 0, f'the segmentation environment does not work: {r.stderr[-800:]}'
    py, tv, cuda, tsv, nnv, npv = r.stdout.split()
    TS_CHECK = {'python': py, 'torch': tv, 'cuda': cuda == 'True', 'TotalSegmentator': tsv, 'nnunetv2': nnv, 'numpy': npv}
    assert (tsv, nnv) == (TS_VERSION, TS_NNUNET) and tv.startswith(TS_TORCH), f'unexpected versions: {TS_CHECK}'
    if IN_COLAB:
        assert TS_CHECK['cuda'], 'TotalSegmentator does not see the GPU: choose a GPU runtime and run all again'
print('TotalSegmentator environment:', TS_CHECK if TS_CHECK else 'not installed (local fake-image test)')

## Settings (plan sections 7d–7j, unchanged, and 7o)

In [ ]:
# 3. Settings (notebook 11, unchanged)
SEED = 2026
N_SPLITS, N_REPEATS = 5, 5
N_BOOT = 2000
N_PERM = 999
MIN_COUNT = 3
FLAG_LOWER_CI, FLAG_Q = 0.60, 0.05
LR_PARAMS = dict(C=1.0, class_weight='balanced', max_iter=5000)
SCANNER_COLS = ['vendor', 'model', 'software_version', 'image_count', 'slice_thickness_mm', 'kernel', 'body_part',
                'protocol_text']
CATEGORICAL = {'vendor', 'model', 'software_version', 'kernel', 'body_part',
               'hdr_kernel', 'hdr_filter_type', 'hdr_patient_position'}
LOG_NUMERIC = {'image_count', 'body_area_cm2'}
TEXT = {'protocol_text', 'care_text'}

TCIA = 'https://services.cancerimagingarchive.net/nbia-api/services/v1'
PREP_JOBS = max(1, min(8, N_CPU - 1))
EMB_WORKERS = max(1, min(6, N_CPU - 2)) if IN_COLAB else 0
BODY_HU, OPEN_MM, DILATE_MM, AIR_HU = -500, 3.0, 5.0, -1000
N_QC = 2
CTFM_REPO, CTFM_REVISION = 'project-lighter/ct_fm_feature_extractor', '527b0d1c262b433396c85ff2de84c427b26945a1'
CTFM_SPACING, CTFM_PATCH, CTFM_OVERLAP, CTFM_BATCH = (3.0, 1.0, 1.0), (24, 128, 128), 0.625, 32
MERLIN_REPO, MERLIN_REVISION = 'stanfordmimi/Merlin', '66dd07585de79e4d8d70d67806da88f3ccaa5a0c'
MERLIN_CKPT = 'i3_resnet_clinical_longformer_best_clip_04-02-2024_23-21-36_epoch_99.pt'
ENCODERS = ('ctfm', 'merlin')
HDR_NUMERIC = ['hdr_slice_thickness', 'hdr_z_spacing', 'hdr_pixel_spacing', 'hdr_recon_diameter', 'hdr_kvp',
               'hdr_tube_current', 'hdr_exposure', 'hdr_ctdivol', 'hdr_pitch']
HDR_CATEGORICAL = ['hdr_kernel', 'hdr_filter_type', 'hdr_patient_position']
HDR_COLS = HDR_NUMERIC + HDR_CATEGORICAL
GATE_NULL_RANGE = (0.35, 0.55)
GATE_MIN_COVERAGE = 0.95
MAX_SERIES_SET_ASIDE = 3   # amendment of 4 Nov 2026: series without CT slices set aside per patient, at most
REPRO_TOL = 1e-3            # as notebooks 06 to 09
SIL_HU = 0                  # notebook 06's silhouette

PREP_KEY05 = hashlib.md5(json.dumps({'BODY_HU': BODY_HU, 'OPEN_MM': OPEN_MM, 'DILATE_MM': DILATE_MM, 'AIR_HU': AIR_HU,
                                     'v': 1}, sort_keys=True).encode()).hexdigest()[:12]
EMB_KEY05 = hashlib.md5(json.dumps({'prep': PREP_KEY05, 'ctfm': [CTFM_REVISION, CTFM_SPACING, CTFM_PATCH, CTFM_OVERLAP],
                                    'merlin': [MERLIN_REVISION, MERLIN_CKPT], 'v': 1},
                                   sort_keys=True).encode()).hexdigest()[:12]
EMB_KEY = hashlib.md5(json.dumps({'nb05': EMB_KEY05, 'SIL_HU': SIL_HU, 'v': 1}, sort_keys=True).encode()).hexdigest()[:12]
PREP_KEY = hashlib.md5(json.dumps({'nb05': PREP_KEY05, 'SIL_HU': SIL_HU, 'nb08': 1},
                                  sort_keys=True).encode()).hexdigest()[:12]   # notebook 08's, unchanged
VARIANTS = ('original', 'background', 'silhouette')       # prepared and embedded, as in notebooks 07 and 08
INPUTS_B = ('original', 'background')                     # notebook 08's transfer analysis
INPUT_NAME = {'original': 'image', 'background': 'background', 'silhouette': 'silhouette'}

# The lung pairs: the source is notebook 05's flagged task; two old targets (notebook 08) and two new ones (plan 7j)
SOURCE = 'lung histology (LUAD vs LUSC)'
SOURCES = [SOURCE]
NSCLC_TASK = 'NSCLC Radiogenomics | histology'
OLD_TARGETS = ('NSCLC Radiogenomics', 'CPTAC lung')
NEW_TARGETS = ('NSCLC-Radiomics', 'Lung-PET-CT-Dx')
PAIR_OF = {'NSCLC Radiogenomics': 'lung: TCGA -> NSCLC Radiogenomics', 'CPTAC lung': 'lung: TCGA -> CPTAC',
           'NSCLC-Radiomics': 'lung: TCGA -> NSCLC-Radiomics', 'Lung-PET-CT-Dx': 'lung: TCGA -> Lung-PET-CT-Dx'}
PAIRS = [{'pair': PAIR_OF[t], 'source': SOURCE, 'target': t} for t in OLD_TARGETS + NEW_TARGETS]
ROLE = {'NSCLC Radiogenomics': 'independent (notebook 08)', 'CPTAC lung': 'one collection per class (notebook 08)',
        'NSCLC-Radiomics': 'independent (new, plan 7j)', 'Lung-PET-CT-Dx': 'independent (new, plan 7j)'}
# one class order for every dataset: (class 0, class 1); class 1 is squamous cell carcinoma
CLASSES = {SOURCE: ('TCGA-LUAD', 'TCGA-LUSC'), 'NSCLC Radiogenomics': ('adenocarcinoma', 'squamous'),
           'CPTAC lung': ('CPTAC-LUAD', 'CPTAC-LSCC'),
           'NSCLC-Radiomics': ('adenocarcinoma', 'squamous cell carcinoma'), 'Lung-PET-CT-Dx': ('A', 'G')}
# counts written into the plan before any image (old targets: notebook 08's plan; new targets: plan 7j A.1)
EXPECTED_TARGETS = {'NSCLC Radiogenomics': {'adenocarcinoma': 166, 'squamous': 33},
                    'CPTAC lung': {'CPTAC-LUAD': 37, 'CPTAC-LSCC': 36}}
EXPECTED_LABELS_NEW = {'NSCLC-Radiomics': {'adenocarcinoma': 51, 'squamous cell carcinoma': 152},
                       'Lung-PET-CT-Dx': {'A': 251, 'G': 61}}
METADATA_TARGETS = ()       # the metadata model is not part of 7j (the new cohorts were not in notebook 02)

# Lung1 clinical file (NSCLC-Radiomics), pinned on 3 Nov 2026
LUNG1_COLLECTION = 'NSCLC-Radiomics'
LUNG1_URL = ('https://www.cancerimagingarchive.net/wp-content/uploads/'
             'NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv')
LUNG1_MD5 = '2d2638a2ac1d34edc6535b59b6f71510'
# Lung-PET-CT-Dx: the class is the letter after "Lung_Dx-" (A adenocarcinoma, B small cell, E large cell, G squamous)
DX_COLLECTION = 'Lung-PET-CT-Dx'
DX_ID = r'Lung_Dx-([ABEG])\d{4}'

# Plan 7j A.6 (descriptive): the fields compared by class in every lung dataset (notebook 09's list)
FIELDS_NUMERIC = ['hdr_slice_thickness', 'hdr_z_spacing', 'hdr_recon_diameter', 'hdr_kvp', 'hdr_tube_current',
                  'body_area_cm2']

# Notebook 01's rule 7a.7 (frozen settings, copied unchanged; the function below is checked against notebook 01's output)
IMAGING_MODALITIES = ('CT', 'MR')
MIN_IMAGES = 10
EXCLUDE_SERIES_PATTERN = (r'scout|locali[sz]er|\bloc\b|topogram|survey|calibration|\bcal\b|'
                          r'bolus|smart\s*prep|monitor|dose\s*(?:report|info)|screen\s*save|'
                          r'secondary|derived|\bmpr\b|\bmip\b|reformat|fusion')
CT_REFORMAT_PATTERN = r'\bsag|\bcor\b|coronal'

# Local code tests only (ignored in Colab)
TEST = {}
if not IN_COLAB:
    TEST = {'patients': [p for p in os.environ.get('ASA_TEST_PATIENTS', '').split(';') if p],
            'fake_images': os.environ.get('ASA_TEST_FAKE_IMAGES') == '1',
            'stop_after_images': os.environ.get('ASA_TEST_STOP_AFTER_IMAGES') == '1',
            'real_is_shuffled': True}   # local tests never see the real labels
    N_PERM = int(os.environ.get('ASA_TEST_N_PERM', N_PERM))
    N_BOOT = int(os.environ.get('ASA_TEST_N_BOOT', N_BOOT))
    CTFM_BATCH = int(os.environ.get('ASA_TEST_CTFM_BATCH', CTFM_BATCH))
    print('LOCAL TEST SETTINGS:', TEST, '| N_PERM', N_PERM, '| N_BOOT', N_BOOT)

In [ ]:
# 3b. Settings of this notebook (plan 7o; fixed before any image was processed)
LIDC_TASK = 'LIDC-IDRI | primary vs metastatic'
LUNG_TARGETS = ('CPTAC lung', 'NSCLC Radiogenomics', 'NSCLC-Radiomics', 'Lung-PET-CT-Dx')
INDEPENDENT = ('NSCLC Radiogenomics', 'NSCLC-Radiomics', 'Lung-PET-CT-Dx')
DATASETS12 = (SOURCE,) + LUNG_TARGETS + (LIDC_TASK,)
LUNG_ROIS = ['lung_upper_lobe_left', 'lung_lower_lobe_left', 'lung_upper_lobe_right', 'lung_middle_lobe_right',
             'lung_lower_lobe_right']
LUNG_DILATE_MM, TUMOUR_DILATE_MM = 5.0, 10.0
GTV_PATTERN = r'^\s*gtv[\s_-]*1\s*$'
TUMOUR_CHECK_DICE, TUMOUR_CHECK_MIN_SHARE = 0.1, 0.70
FAT_HU, FAT_RING_MM, ARM_MIN_CM2 = (-190, -30), 30.0, 10.0
HARM_INPLANE_MM, HARM_SLAB_MM, HARM_FOV_MM, HARM_Z_MARGIN_MM = 1.0, 5.0, 360.0, 20.0
CNN_INPLANE_MM, CNN_SHAPE = 2.0, (64, 180, 180)             # (z slabs of 5 mm, y, x)
REPRO_BODY_TOL = 1e-6
EMB_REPRO_MIN_COS = 0.999
VARIANTS12 = ('original', 'lungs', 'tumour')               # embedded with the original's CT-FM crop box
HARM_VARIANTS = ('harm_full', 'harm_background')            # embedded with harm_full's crop box
INPUTS_G1 = ('lungs', 'tumour')
INPUTS_G3 = ('harm_background', 'harm_full')
HABITUS = ['ap_lr_ratio', 'subcut_fat_cm2', 'arms_beside_trunk', 'table_height_mm', 'centroid_offset_mm',
           'scan_length_mm', 'lung_volume_l']
N_PERM_STRICT = 99999
TS_TIMEOUT = 1800                                          # seconds per TotalSegmentator call (about 1 min on a GPU)
if TEST:                                                  # local code tests only
    N_PERM_STRICT = int(os.environ.get('ASA_TEST_N_PERM_STRICT', 999))
PREP12_KEY = hashlib.md5(json.dumps({'prep08': PREP_KEY, 'ts': [TS_VERSION, TS_TORCH, TS_NNUNET], 'rois': LUNG_ROIS, 'lung_mm': LUNG_DILATE_MM,
                                     'tum_mm': TUMOUR_DILATE_MM, 'fat': FAT_HU, 'ring': FAT_RING_MM, 'arm': ARM_MIN_CM2,
                                     'harm': [HARM_INPLANE_MM, HARM_SLAB_MM, HARM_FOV_MM, HARM_Z_MARGIN_MM],
                                     'cnn': [CNN_INPLANE_MM, CNN_SHAPE], 'v': 1}, sort_keys=True).encode()).hexdigest()[:12]
EMB12_KEY = hashlib.md5(json.dumps({'prep12': PREP12_KEY, 'emb': EMB_KEY, 'v': 1}, sort_keys=True).encode()).hexdigest()[:12]
print('PREP12_KEY', PREP12_KEY, '| EMB12_KEY', EMB12_KEY)

In [ ]:
# 4. Load and verify the inputs from notebooks 01-09; the source and the four lung targets (notebook 11, unchanged)
def verify(folder):
    man = json.loads((folder / 'run_manifest.json').read_text())
    bad = [f['file'] for f in man['outputs'] if md5(folder / f['file']) != f['md5']]
    assert not bad, f'These files in {folder} changed since that run: {bad}'
    return man

def cache_md5(folder):
    files = sorted((q for q in folder.rglob('*') if q.is_file() and q.suffix in ('.npz', '.json', '.csv')),
                   key=lambda q: q.relative_to(folder).parts)
    return len(files), hashlib.md5(''.join(md5(q) for q in files).encode()).hexdigest()

man01, man02, man03, man04 = verify(IN01), verify(IN02), verify(IN03), verify(IN04)
man = {nb: {m: verify(d / m) for m in ('rehearsal', 'real')} for nb, d in (('05', IN05), ('06', IN06), ('07', IN07),
                                                                          ('08', IN08), ('09', IN09))}
runs = {'01': man01['run_started_utc'], '02': man02['run_started_utc'], '03': man03['run_started_utc'],
        '04': man04['run_started_utc']}
for nb in ('05', '06', '07', '08', '09'):
    for m in ('rehearsal', 'real'):
        assert man[nb][m]['input_runs'] == runs, f'notebook {nb} ({m}) was run on other inputs'
        if not TEST:
            assert man[nb][m]['in_colab'] and not man[nb][m]['local_test_settings'], f'notebook {nb} {m} is not the Colab run'
    runs = {**runs, nb: man[nb]['real']['run_started_utc']}
for nb in ('05', '07', '08'):
    assert man[nb]['real']['labels'] == 'real' and man[nb]['rehearsal']['labels'].startswith('shuffled'), nb
s05 = man['05']['real']['settings']
same = {'SEED': SEED, 'N_SPLITS': N_SPLITS, 'N_REPEATS': N_REPEATS, 'MIN_COUNT': MIN_COUNT,
        'FLAG_LOWER_CI': FLAG_LOWER_CI, 'FLAG_Q': FLAG_Q, 'LR_PARAMS': LR_PARAMS, 'BODY_HU': BODY_HU,
        'OPEN_MM': OPEN_MM, 'DILATE_MM': DILATE_MM, 'AIR_HU': AIR_HU, 'HDR_COLS': HDR_COLS, 'PREP_KEY': PREP_KEY05,
        'EMB_KEY': EMB_KEY05, 'CTFM': [CTFM_REPO, CTFM_REVISION, list(CTFM_SPACING), list(CTFM_PATCH), CTFM_OVERLAP],
        'MERLIN': [MERLIN_REPO, MERLIN_REVISION, MERLIN_CKPT]}
if not TEST:
    same.update({'N_BOOT': N_BOOT, 'N_PERM': N_PERM})
diff = [k for k, v in same.items() if json.loads(json.dumps(s05[k])) != json.loads(json.dumps(v))]
assert not diff, f'settings differ from notebook 05: {diff}'
s08 = man['08']['real']['settings']
assert (s08['SIL_HU'], s08['EMB_KEY08'], s08['PREP_KEY08']) == (SIL_HU, EMB_KEY, PREP_KEY), 'settings differ from notebook 08'
assert list(s08['INPUTS_B']) == list(INPUTS_B), 'notebook 08 analysed other inputs'
assert all(man['08']['real']['classes'][k] == list(CLASSES[k]) for k in (SOURCE,) + OLD_TARGETS), \
    'class order differs from notebook 08'
caches = {}
for nb, folder in (('05', CACHE05), ('06', CACHE06), ('07', CACHE07), ('08', CACHE08)):
    n, h = cache_md5(folder)
    assert (n, h) == (man[nb]['real']['cache_files']['count'], man[nb]['real']['cache_files']['combined_md5']), \
        f'notebook {nb} cache differs from its run record'
    caches[nb] = {'count': n, 'combined_md5': h}
v05 = man['05']['real']['versions']
same_versions = (v05['numpy'], v05['pandas'], v05['scipy'], v05['scikit-learn']) == \
                (np.__version__, pd.__version__, scipy.__version__, sklearn.__version__)
print('Inputs verified: notebooks 01-09 and the caches of notebooks 05-08 '
      f"({', '.join(f'{k}: {v['count']} files' for k, v in caches.items())})")
print('Package versions', 'match notebook 05' if same_versions else
      f"DIFFER from notebook 05 (then numpy {v05['numpy']}, pandas {v05['pandas']}, scipy {v05['scipy']}, "
      f"scikit-learn {v05['scikit-learn']}): small differences in the fitted models are possible")

labels = pd.read_csv(IN02 / 'labels_long.csv')
feat0 = pd.read_csv(IN02 / 'patient_features.csv', low_memory=False)
toks0 = pd.read_csv(IN02 / 'patient_tokens.csv.gz')
sites = pd.read_csv(IN02 / 'patient_sites.csv')
main = pd.read_csv(IN01 / 'main_series.csv', low_memory=False)
smeta = pd.read_csv(IN01 / 'series_metadata.csv.gz', usecols=['SeriesInstanceUID', 'FileSize']).drop_duplicates(
    'SeriesInstanceUID')
KEY = ['collection', 'PatientID', 'modality']
text = (toks0.assign(group=np.where(toks0.category == 'care pathway', 'care_text', 'protocol_text'))
             .groupby(KEY + ['group']).token.agg(lambda s: ' '.join(sorted(set(s)))).unstack('group').reset_index())
for c in ('protocol_text', 'care_text'):
    if c not in text:
        text[c] = ''
feat = feat0.merge(text, on=KEY, how='left')
for c in ('protocol_text', 'care_text'):
    feat[c] = feat[c].fillna('')
for c in ('vendor', 'model', 'software_version', 'kernel', 'body_part'):
    feat[c] = feat[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
site_of = sites.drop_duplicates('PatientID').set_index('PatientID').institution
POLISH_WORDS = set(man04['word_lists']['POLISH_WORDS'])
polish = toks0[toks0.token.isin(POLISH_WORDS)].drop_duplicates(KEY)[KEY].assign(polish=True)
a03 = pd.read_csv(IN03 / 'audit_primary.csv').set_index('task_id')
assert a03.flagged[SOURCE] and SOURCE in man['05']['real']['tasks'], 'source task changed'

inv = {nb: pd.read_csv(folder / 'image_inventory.csv') for nb, folder in (('05', CACHE05), ('07', CACHE07),
                                                                            ('08', CACHE08))}
IN05_KEYS, IN07_KEYS, IN08_KEYS = (set(inv[nb][inv[nb].embedded].key) for nb in ('05', '07', '08'))

def main_series_rule(series):
    """One main series per patient and modality: the earliest study; in it, the eligible series with the most images."""
    series = series.replace(r'^\s*$', np.nan, regex=True)
    series['ImageCount'] = pd.to_numeric(series['ImageCount'], errors='coerce')
    series['StudyDate_dt'] = pd.to_datetime(series['StudyDate'], errors='coerce')
    img = series[series.Modality.isin(IMAGING_MODALITIES)].copy()
    desc = img.SeriesDescription.where(img.SeriesDescription.notna(), img.ProtocolName).fillna('')
    img['excluded_pattern'] = desc.str.contains(EXCLUDE_SERIES_PATTERN, flags=re.I, regex=True)
    img['excluded_ct_reformat'] = (img.Modality == 'CT') & desc.str.contains(CT_REFORMAT_PATTERN, flags=re.I, regex=True)
    img['excluded_third_party'] = (img.ThirdPartyAnalysis.astype(str).str.strip().str.lower() == 'yes'
                                   if 'ThirdPartyAnalysis' in img else False)
    img['excluded_small'] = img.ImageCount.fillna(0) < MIN_IMAGES
    img['eligible'] = ~(img.excluded_pattern | img.excluded_ct_reformat | img.excluded_third_party | img.excluded_small)
    pts_by_mod = img.groupby(['Collection', 'Modality']).PatientID.nunique().unstack(fill_value=0)
    primary_modality = {c: ('CT' if r.get('CT', 0) >= r.get('MR', 0) else 'MR') for c, r in pts_by_mod.iterrows()}
    main_rows, no_main = [], []
    for (col, mod, pid), g in img.groupby(['Collection', 'Modality', 'PatientID'], sort=True):
        order = (g.groupby('StudyInstanceUID').StudyDate_dt.min().reset_index()
                  .sort_values(['StudyDate_dt', 'StudyInstanceUID'], na_position='last'))
        for rank, suid in enumerate(order.StudyInstanceUID, 1):
            e = g[(g.StudyInstanceUID == suid) & g.eligible]
            if len(e):
                b = e.sort_values(['ImageCount', 'SeriesNumber', 'SeriesInstanceUID'],
                                  ascending=[False, True, True]).iloc[0]
                main_rows.append({'collection': col, 'PatientID': pid, 'modality': mod,
                                  'is_primary_modality': mod == primary_modality[col],
                                  'StudyInstanceUID': suid, 'StudyDate': b.StudyDate,
                                  'study_rank_used': rank, 'n_studies_this_modality': len(order),
                                  'n_series_this_modality': len(g),
                                  'SeriesInstanceUID': b.SeriesInstanceUID, 'SeriesNumber': b.SeriesNumber,
                                  'SeriesDescription': b.SeriesDescription, 'ProtocolName': b.ProtocolName,
                                  'Manufacturer': b.Manufacturer, 'ManufacturerModelName': b.ManufacturerModelName,
                                  'SoftwareVersions': b.SoftwareVersions, 'ImageCount': b.ImageCount})
                break
        else:
            no_main.append({'collection': col, 'PatientID': pid, 'modality': mod})
    return pd.DataFrame(main_rows), pd.DataFrame(no_main, columns=['collection', 'PatientID', 'modality'])

check, _ = main_series_rule(pd.read_csv(IN01 / 'series_metadata.csv.gz', low_memory=False))
cmp = main[KEY + ['SeriesInstanceUID']].merge(check[KEY + ['SeriesInstanceUID']], on=KEY, how='outer',
                                              suffixes=('', '_rule'))
RULE_REPRODUCED = bool(len(check) == len(main) and (cmp.SeriesInstanceUID == cmp.SeriesInstanceUID_rule).all())
assert RULE_REPRODUCED, 'the main-series function does not reproduce notebook 01'
print(f'Main-series rule: reproduces notebook 01 for all {len(main):,} patient-modality pairs')

def tcia_json(endpoint, **params):
    """A TCIA API reply (series metadata, no images), cached on Drive so that reruns use the same reply."""
    path = CACHE / 'api' / f"{endpoint}__{'__'.join(f'{k}={v}' for k, v in sorted(params.items()))}.json"
    if not path.exists():
        last = None
        for wait in (0, 10, 30, 90):
            time.sleep(wait)
            try:
                r = requests.get(f'{TCIA}/{endpoint}', params=params, timeout=(30, 300))
                r.raise_for_status()
                data = r.json()
                assert len(data), 'empty reply'
                write_json_atomic(path, {'request': {'endpoint': endpoint, **params}, 'fetched_utc':
                                         datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
                                         'data': data})
                break
            except Exception as e:
                last = e
        else:
            raise RuntimeError(f'TCIA {endpoint} failed 4 times: {last!r}')
    return json.loads(path.read_text())['data'], md5(path)

# old target 1: NSCLC Radiogenomics (notebook 02's task; embedded in notebook 07), as notebook 08 built it
nsclc = labels[labels.task_id == NSCLC_TASK].copy()
nsclc['key'] = nsclc.collection + '__' + nsclc.PatientID
# old target 2: CPTAC lung (every patient with a CT main series in notebook 01; the class is the collection)
cptac = main[main.collection.isin(CLASSES['CPTAC lung']) & (main.modality == 'CT')].copy()
assert not cptac.PatientID.duplicated().any(), 'a patient is in both CPTAC lung collections'
cptac['label'] = cptac.collection
cptac = cptac.merge(smeta, on='SeriesInstanceUID', how='left')

# new target 1: NSCLC-Radiomics, labels from the pinned Lung1 clinical file
lung1_path = LABEL_DIR / 'NSCLC-Radiomics-Lung1.clinical-version3-Oct-2019.csv'
if not lung1_path.exists():
    r = requests.get(LUNG1_URL, timeout=(30, 120))
    r.raise_for_status()
    lung1_path.write_bytes(r.content)
assert md5(lung1_path) == LUNG1_MD5, 'the Lung1 clinical file differs from the pinned one'
lung1 = pd.read_csv(lung1_path)
lung1 = lung1[lung1.Histology.isin(CLASSES['NSCLC-Radiomics'])][['PatientID', 'Histology']].rename(
    columns={'Histology': 'label'})
# new target 2: Lung-PET-CT-Dx, the class from the patient-ID code
NEW_FRAMES, API_MD5, NO_SERIES, SERIES_NEW = {}, {}, {}, {}
for coll in (LUNG1_COLLECTION, DX_COLLECTION):
    reply, API_MD5[coll] = tcia_json('getSeries', Collection=coll)
    series = pd.DataFrame(reply)
    SERIES_NEW[coll] = series            # kept for the amendment of 4 Nov 2026 (series without CT slices)
    m_ct, _ = main_series_rule(series)
    m_ct = m_ct[m_ct.modality == 'CT']
    sizes = series.drop_duplicates('SeriesInstanceUID').set_index('SeriesInstanceUID').FileSize.astype(float)
    if coll == LUNG1_COLLECTION:
        lab = lung1
    else:
        ids = pd.Series(series.PatientID.unique())
        assert ids.str.fullmatch(DX_ID).all(), 'a Lung-PET-CT-Dx patient ID does not follow the documented code'
        lab = pd.DataFrame({'PatientID': ids, 'label': ids.str.extract(DX_ID, expand=False)})
        lab = lab[lab.label.isin(CLASSES['Lung-PET-CT-Dx'])]
    name = 'NSCLC-Radiomics' if coll == LUNG1_COLLECTION else 'Lung-PET-CT-Dx'
    counts = lab.label.value_counts().to_dict()
    assert counts == EXPECTED_LABELS_NEW[name], f'{name}: label counts {counts} differ from plan 7j {EXPECTED_LABELS_NEW[name]}'
    df = lab.merge(m_ct, on='PatientID', how='left')
    NO_SERIES[name] = df[df.SeriesInstanceUID.isna()].PatientID.tolist()
    df = df[df.SeriesInstanceUID.notna()].copy()
    df['collection'] = coll
    df['FileSize'] = df.SeriesInstanceUID.map(sizes)
    NEW_FRAMES[name] = df

TARGETS = {}
for name, df in (('NSCLC Radiogenomics', nsclc), ('CPTAC lung', cptac), *NEW_FRAMES.items()):
    df = df.copy()
    df['key'] = df.collection + '__' + df.PatientID
    df['modality'] = 'CT'
    assert set(df.label) <= set(CLASSES[name]), f'{name}: unexpected labels {sorted(set(df.label))}'
    if name in EXPECTED_TARGETS:
        counts = df.label.value_counts().to_dict()
        assert counts == EXPECTED_TARGETS[name], f'{name}: {counts} differs from notebook 08 {EXPECTED_TARGETS[name]}'
    TARGETS[name] = df.sort_values('key').reset_index(drop=True)
assert set(TARGETS['NSCLC Radiogenomics'].key) <= IN07_KEYS, 'an NSCLC Radiogenomics patient is not embedded in notebook 07'
assert set(TARGETS['CPTAC lung'].key) <= IN08_KEYS, 'a CPTAC lung patient is not embedded in notebook 08'

new = pd.concat([TARGETS[t].assign(target=t) for t in NEW_TARGETS], ignore_index=True)
assert not new.key.isin(IN05_KEYS | IN07_KEYS | IN08_KEYS).any(), 'a new target patient was already embedded'
PATIENTS = new.sort_values('key').reset_index(drop=True)
NEW = PATIENTS
if TEST.get('patients'):
    NEW = NEW[NEW.key.isin(TEST['patients'])].reset_index(drop=True)
QC_KEYS = set(NEW.sort_values('key').groupby('target').key.head(N_QC))
print('Targets: ' + '; '.join(f"{t} {len(d)} ({' / '.join(f'{c} {n}' for c, n in d.label.value_counts().items())})"
                              for t, d in TARGETS.items()))
print('New-target patients without an eligible CT main series (dropped, reported):',
      {k: len(v) for k, v in NO_SERIES.items()})
print(f'{len(PATIENTS)} new patients ({PATIENTS.FileSize.sum() / 1e9:.1f} GB of DICOM to download)')

## Part 1 · Download, segment and embed (no labels used)

In [ ]:
# 5. Download, read and convert one series (notebook 08, unchanged)
def download_series(uid, dest):
    last = None
    for wait in (0, 10, 30, 90, 240):
        time.sleep(wait)
        try:
            tmp = dest.with_name(dest.name + '.part')
            with requests.get(f'{TCIA}/getImage', params={'SeriesInstanceUID': uid}, stream=True,
                              timeout=(30, 600)) as r:
                r.raise_for_status()
                with open(tmp, 'wb') as f:
                    for chunk in r.iter_content(1 << 20):
                        f.write(chunk)
            with zipfile.ZipFile(tmp) as z:
                badf = z.testzip()
                assert badf is None, f'corrupt file in zip: {badf}'
            tmp.replace(dest)
            return dest.stat().st_size
        except Exception as e:   # network errors, partial files: try again
            last = e
    raise RuntimeError(f'download failed 5 times: {last!r}')

def _num(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan

def _text(v):
    if v is None:
        return np.nan
    if isinstance(v, pydicom.multival.MultiValue):
        v = '/'.join(str(x) for x in v)
    v = str(v).strip().upper()
    return v if v else np.nan

def read_series(folder, uid):
    """The slices of one series, sorted along the slice normal, with geometry checks."""
    metas = []
    for p in sorted(folder.rglob('*')):
        if not p.is_file():
            continue
        try:
            ds = pydicom.dcmread(p, stop_before_pixels=True)
        except Exception:
            continue
        if str(getattr(ds, 'SeriesInstanceUID', '')) != uid:
            continue
        if not all(t in ds for t in ('ImagePositionPatient', 'ImageOrientationPatient', 'PixelSpacing', 'Rows')):
            continue
        metas.append((p, ds))
    assert metas, 'no image slices in the download'
    combo = [(tuple(np.round(np.asarray(ds.ImageOrientationPatient, float), 3)), int(ds.Rows), int(ds.Columns))
             for _, ds in metas]
    major = pd.Series(combo).value_counts().index[0]
    keep = [m for m, c in zip(metas, combo) if c == major]
    iop = np.asarray(keep[0][1].ImageOrientationPatient, float)
    row_cos, col_cos = iop[:3], iop[3:]
    normal = np.cross(row_cos, col_cos)
    pos = np.array([[float(v) for v in ds.ImagePositionPatient] for _, ds in keep])
    proj = pos @ normal
    inst = np.array([_num(getattr(ds, 'InstanceNumber', np.nan)) for _, ds in keep])
    order = np.lexsort((np.nan_to_num(inst, nan=1e9), proj))
    dup = np.r_[False, np.diff(proj[order]) < 1e-3]          # same position twice: keep the first
    order = order[~dup]
    keep, pos, proj = [keep[i] for i in order], pos[order], proj[order]
    assert len(keep) >= 10, f'only {len(keep)} usable slices'
    gaps = np.diff(proj)
    z = float(np.median(gaps))
    step = pos[-1] - pos[0]
    geo = {'row_cos': row_cos, 'col_cos': col_cos, 'normal': normal, 'origin': pos[0], 'z': z,
           'n_other_orientation_dropped': sum(c != major for c in combo),
           'n_duplicates_dropped': int(dup.sum()),
           'max_gap_deviation_mm': float(np.max(np.abs(gaps - z))),
           'nonuniform_spacing': bool(np.max(np.abs(gaps - z)) > max(0.05, 0.1 * z)),
           'gantry_tilt_deg': float(np.degrees(np.arccos(min(1.0, abs(step @ normal) / np.linalg.norm(step))))),
           'coverage_mm': float(proj[-1] - proj[0] + z)}
    return keep, geo

def header_fields(keep, geo):
    ds0 = keep[0][1]
    def med(name):
        vals = [_num(getattr(ds, name, None)) for _, ds in keep]
        vals = [v for v in vals if np.isfinite(v)]
        return float(np.median(vals)) if vals else np.nan
    ps = [float(v) for v in ds0.PixelSpacing]
    agent = getattr(ds0, 'ContrastBolusAgent', None)
    return {'hdr_manufacturer': _text(getattr(ds0, 'Manufacturer', None)),
            'hdr_model': _text(getattr(ds0, 'ManufacturerModelName', None)),
            'hdr_slice_thickness': med('SliceThickness'), 'hdr_z_spacing': geo['z'],
            'hdr_pixel_spacing': float(np.mean(ps)), 'hdr_rows': int(ds0.Rows), 'hdr_columns': int(ds0.Columns),
            'hdr_n_slices': len(keep), 'hdr_coverage_mm': geo['coverage_mm'],
            'hdr_recon_diameter': med('ReconstructionDiameter'), 'hdr_kvp': med('KVP'),
            'hdr_tube_current': med('XRayTubeCurrent'), 'hdr_exposure': med('Exposure'),
            'hdr_ctdivol': med('CTDIvol'), 'hdr_pitch': med('SpiralPitchFactor'),
            'hdr_kernel': _text(getattr(ds0, 'ConvolutionKernel', None)),
            'hdr_filter_type': _text(getattr(ds0, 'FilterType', None)),
            'hdr_patient_position': _text(getattr(ds0, 'PatientPosition', None)),
            'hdr_contrast_agent_given': ('yes' if agent is not None and str(agent).strip() else 'no'),
            'hdr_gantry_tilt_deg': geo['gantry_tilt_deg'], 'hdr_nonuniform_spacing': geo['nonuniform_spacing'],
            'hdr_max_gap_deviation_mm': geo['max_gap_deviation_mm'],
            'hdr_duplicates_dropped': geo['n_duplicates_dropped'],
            'hdr_other_orientation_dropped': geo['n_other_orientation_dropped']}

def to_image(keep, geo):
    """HU volume with the geometry from the DICOM positions (not from the reader's guess)."""
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames([str(p) for p, _ in keep])
    reader.SetOutputPixelType(sitk.sitkFloat32)          # GDCM applies the rescale slope and intercept (HU)
    img = reader.Execute()
    ps = [float(v) for v in keep[0][1].PixelSpacing]      # [between rows, between columns]
    img.SetSpacing((ps[1], ps[0], geo['z']))
    img.SetOrigin(tuple(float(v) for v in geo['origin']))
    img.SetDirection(tuple(np.stack([geo['row_cos'], geo['col_cos'], geo['normal']], axis=1).ravel()))
    return img

def like(arr, ref, dtype=np.int16):
    """A SimpleITK image from a (z, y, x) array, with the geometry of `ref`."""
    if dtype == np.int16:
        arr = np.clip(np.round(arr), -32768, 32767)
    out = sitk.GetImageFromArray(arr.astype(dtype))
    out.CopyInformation(ref)
    return out

In [ ]:
# 6. Body mask, background-only version, silhouette and QC picture (notebook 08, unchanged)
def body_mask(arr, sp):
    """arr: (z, y, x) HU; sp: (sz, sy, sx) mm. Returns (body, body dilated by DILATE_MM)."""
    m = arr > BODY_HU
    r = max(1, int(round(OPEN_MM / ((sp[1] + sp[2]) / 2))))
    yy, xx = np.mgrid[-r:r + 1, -r:r + 1]
    m = ndimage.binary_opening(m, structure=(yy ** 2 + xx ** 2 <= r * r)[None])
    lab_, n = ndimage.label(m)
    if n == 0:
        return np.zeros_like(m), np.zeros_like(m)
    counts = np.bincount(lab_.ravel())
    counts[0] = 0
    largest = lab_ == counts.argmax()
    del lab_, m
    core = ndimage.binary_fill_holes(largest, structure=np.ones((1, 3, 3), bool))
    del largest
    # dilation by DILATE_MM, computed inside the bounding box plus a margin (same result, much faster)
    pad = [int(np.ceil(DILATE_MM / s)) + 1 for s in sp]
    idx = np.where(core)
    lo = [max(0, int(i.min()) - p) for i, p in zip(idx, pad)]
    hi = [min(n_, int(i.max()) + p + 1) for i, n_, p in zip(idx, core.shape, pad)]
    box = tuple(slice(a, b) for a, b in zip(lo, hi))
    dil = np.zeros_like(core)
    dil[box] = ndimage.distance_transform_edt(~core[box], sampling=sp).astype(np.float32) <= DILATE_MM
    return core, dil

def silhouette(dil):
    """Notebook 06: the region the background-only version blanked, as a flat shape; air outside."""
    return np.where(dil, SIL_HU, AIR_HU).astype(np.int16)

def qc_picture(key, arr, core, bg, sil, path):
    from matplotlib.figure import Figure
    z = int(np.argmax(core.sum(axis=(1, 2))))
    fig = Figure(figsize=(12, 4.2))
    ax = fig.subplots(1, 3)
    kw = dict(cmap='gray', vmin=-1000, vmax=400)
    ax[0].imshow(arr[z], **kw); ax[0].contour(core[z], levels=[0.5], colors='r', linewidths=0.6)
    ax[0].set_title('original + body outline')
    ax[1].imshow(bg[z], **kw); ax[1].set_title('background only')
    ax[2].imshow(sil[z], **kw); ax[2].set_title('silhouette')
    for a in ax:
        a.axis('off')
    fig.suptitle(key, fontsize=9)
    fig.tight_layout()
    fig.savefig(path, dpi=80)

def nifti_paths(key):
    d = WORK / 'nifti'
    return {v: d / f'{key}__{v}.nii.gz' for v in VARIANTS}

def header_path(key):
    return CACHE / 'headers' / f'{key}.json'

def header_ok(key):
    p = header_path(key)
    return p.exists() and json.loads(p.read_text()).get('prep_key') == PREP_KEY

In [ ]:
# 7. Load the two encoders (notebook 08, unchanged)
i3res.checkpoint = type('NoCheckpoint', (), {'checkpoint': staticmethod(lambda fn, *a, **k: fn(*a))})

def load_ctfm():
    cfg = json.loads(Path(hf_hub_download(CTFM_REPO, 'config.json', revision=CTFM_REVISION)).read_text())
    model = SegResEncoder(**cfg)
    state = safetensors.torch.load_file(hf_hub_download(CTFM_REPO, 'model.safetensors', revision=CTFM_REVISION))
    model.load_state_dict(state, strict=True)
    return model.eval()

MERLIN_DIR = Path(merlin.__file__).parent / 'models' / 'checkpoints'   # where the official class looks for it
if not IN_COLAB and os.environ.get('ASA_TEST_MERLIN_DIR'):              # local tests: Windows path-length limit
    MERLIN_DIR = Path(os.environ['ASA_TEST_MERLIN_DIR'])

def load_merlin():
    ckpt = hf_hub_download(MERLIN_REPO, MERLIN_CKPT, revision=MERLIN_REVISION, local_dir=MERLIN_DIR)
    state = torch.load(ckpt, map_location='cpu')
    prefix = 'encode_image.i3_resnet.'
    enc = i3res.I3ResNet(copy.deepcopy(torchvision.models.resnet152(weights=None)), class_nb=1692, conv_class=True,
                         ImageEmbedding=True)
    enc.load_state_dict({k[len(prefix):]: v for k, v in state.items() if k.startswith(prefix)}, strict=True)
    return enc.eval()

# CT-FM preprocessing (authors' script), split so that the background-only version can use the original's crop box
CTFM_LOAD = Compose([LoadImage(ensure_channel_first=True), EnsureType(), Orientation(axcodes='SPL'),
                     Spacing(pixdim=CTFM_SPACING, mode='bilinear')])
CTFM_SCALE = ScaleIntensityRange(a_min=-1024, a_max=2048, b_min=0, b_max=1, clip=True)

def ctfm_inputs(paths, crop_from=None):
    """Preprocessed CT-FM inputs for each variant. With crop_from, every variant uses that variant's foreground box."""
    imgs = {v: CTFM_LOAD(str(p)) for v, p in paths.items()}
    out, boxes = {}, {}
    for v, img in imgs.items():
        src = imgs[crop_from] if crop_from else img
        start, end = generate_spatial_bounding_box(src, select_fn=lambda x: x > 0)   # CropForeground's default
        out[v] = _plain(CTFM_SCALE(SpatialCrop(roi_start=start, roi_end=end)(img)))
        boxes[v] = [int(a) for a in start] + [int(b) for b in end]
    return out, boxes

def _plain(t):
    return (t.as_tensor() if hasattr(t, 'as_tensor') else torch.as_tensor(t)).float()

def merlin_inputs(paths):
    return {v: _plain(MERLIN_TRANSFORMS({'image': str(p)})['image']) for v, p in paths.items()}

class PrepDataset(torch.utils.data.Dataset):
    """CPU preprocessing in DataLoader workers; the GPU only runs the encoders."""
    def __init__(self, items):
        self.items = items              # (key, {variant: path}, crop_from)
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        key, paths, crop_from = self.items[i]
        try:
            ct, boxes = ctfm_inputs(paths, crop_from)
            return {'key': key, 'ctfm': ct, 'merlin': merlin_inputs(paths), 'boxes': boxes, 'error': None}
        except Exception as e:
            root = e
            while root.__cause__ is not None:      # MONAI wraps the real error; report the root cause
                root = root.__cause__
            return {'key': key, 'error': f'{e!r} | root cause: {root!r}'[:600]}

@torch.no_grad()
def ctfm_embed(model, vol):
    feats, batch = [], []
    def run(b):
        x = torch.cat(b).to(DEVICE)
        return torch.nn.functional.adaptive_avg_pool3d(model(x)[-1], 1).flatten(1).cpu()
    for patch, _ in SlidingWindowSplitter(CTFM_PATCH, CTFM_OVERLAP)(vol.unsqueeze(0)):
        batch.append(patch)
        if len(batch) == CTFM_BATCH:
            feats.append(run(batch)); batch = []
    if batch:
        feats.append(run(batch))
    f = torch.cat(feats)
    return f.mean(0).numpy().astype(np.float32), int(len(f))

@torch.no_grad()
def merlin_embed(model, vol):
    return model(vol.unsqueeze(0).to(DEVICE)).reshape(-1).cpu().numpy().astype(np.float32)

def embed_items(items, out_path_of, models, extra_of=None, label='scans'):
    """Embed every item with both encoders and save one .npz per item (resume point)."""
    loader = torch.utils.data.DataLoader(PrepDataset(items), batch_size=None, shuffle=False,
                                         num_workers=EMB_WORKERS if len(items) > 1 else 0)
    done, failed, t0 = 0, [], time.time()
    for rec in loader:
        if rec['error']:
            failed.append({'key': rec['key'], 'error': rec['error']})
            continue
        arrays, meta = {}, {'emb_key': EMB_KEY, 'boxes': rec['boxes'], **(extra_of(rec['key']) if extra_of else {})}
        for v, vol in rec['ctfm'].items():
            arrays[f'ctfm__{v}'], meta[f'ctfm__{v}__patches'] = ctfm_embed(models['ctfm'], vol)
        for v, vol in rec['merlin'].items():
            arrays[f'merlin__{v}'] = merlin_embed(models['merlin'], vol)
        path = out_path_of(rec['key'])
        tmp = path.with_name(path.stem + '.tmp.npz')
        np.savez(tmp, meta=json.dumps(meta, default=_jsonable), **arrays)
        tmp.replace(path)
        done += 1
        if done % 25 == 0 or done == len(items):
            print(f'  {label}: {done}/{len(items)} embedded ({(time.time() - t0) / 60:.1f} min)')
    return failed

def emb_path(key):
    return CACHE / 'embeddings' / f'{key}.npz'

def emb_ok(path, variants):
    if not path.exists():
        return False
    with np.load(path) as z:
        meta = json.loads(str(z['meta']))
        return meta.get('emb_key') == EMB_KEY and all(f'{e}__{v}' in z.files for e in ENCODERS for v in variants)

def emb_file(key, variant):
    """Where a scan's embedding is: notebook 05 (original, background) or 06 (silhouette), notebook 07, or this notebook."""
    if key in IN05_KEYS:
        return (CACHE06 if variant == 'silhouette' else CACHE05) / 'embeddings' / f'{key}.npz'
    if key in IN07_KEYS:
        return CACHE07 / 'embeddings' / f'{key}.npz'
    return emb_path(key)

def has_variant(key, variant):
    p = emb_file(key, variant)
    if not p.exists():
        return False
    with np.load(p) as z:
        ok_key = json.loads(str(z['meta'])).get('emb_key') in (EMB_KEY05, EMB_KEY)
        return ok_key and all(f'{e}__{variant}' in z.files for e in ENCODERS)

def header_of(key):
    folder = CACHE05 if key in IN05_KEYS else CACHE07 if key in IN07_KEYS else CACHE
    p = folder / 'headers' / f'{key}.json'
    return json.loads(p.read_text()) if p.exists() else {'key': key}

In [ ]:
# 8. Where each earlier embedding and header record is (notebook 11, unchanged)
def emb_file(key, variant):
    """Where a scan's embedding is: notebook 05 (original, background) or 06 (silhouette), notebook 07, 08, or this one."""
    if key in IN05_KEYS:
        return (CACHE06 if variant == 'silhouette' else CACHE05) / 'embeddings' / f'{key}.npz'
    if key in IN07_KEYS:
        return CACHE07 / 'embeddings' / f'{key}.npz'
    if key in IN08_KEYS:
        return CACHE08 / 'embeddings' / f'{key}.npz'
    return emb_path(key)

def has_variant(key, variant):
    p = emb_file(key, variant)
    if not p.exists():
        return False
    with np.load(p) as z:
        ok_key = json.loads(str(z['meta'])).get('emb_key') in (EMB_KEY05, EMB_KEY)
        return ok_key and all(f'{e}__{variant}' in z.files for e in ENCODERS)

def header_of(key):
    folder = CACHE05 if key in IN05_KEYS else CACHE07 if key in IN07_KEYS else CACHE08 if key in IN08_KEYS else CACHE
    p = folder / 'headers' / f'{key}.json'
    return json.loads(p.read_text()) if p.exists() else {'key': key}


In [ ]:
# 4b. The scans of this notebook: exactly those embedded before, with the series recorded in their header records
DEFS_FROM_11 = None   # (emb_file, has_variant and header_of are notebook 11's, defined above)
rows = []
for name in DATASETS12:
    if name in TARGETS:
        keys = TARGETS[name].key.tolist()
    else:
        d = labels[labels.task_id == name]
        keys = (d.collection + '__' + d.PatientID).tolist()
    for k in sorted(keys):
        h = header_of(k)
        rows.append({'dataset': name, 'key': k, 'collection': k.split('__')[0], 'PatientID': k.split('__')[1],
                     'SeriesInstanceUID': h.get('SeriesInstanceUID'), 'tcia_file_size': h.get('tcia_file_size'),
                     'ref_body_fraction': h.get('body_fraction_dilated'), 'ref_body_area': h.get('body_area_cm2'),
                     'ref_coverage_mm': h.get('hdr_coverage_mm')})
SCANS = pd.DataFrame(rows)
assert not SCANS.key.duplicated().any(), 'a scan is in two datasets'
NO_HEADER = SCANS[SCANS.SeriesInstanceUID.isna()]
SCANS = SCANS[SCANS.SeriesInstanceUID.notna()].reset_index(drop=True)
# the manual tumour contours (GTV-1) of NSCLC-Radiomics: the patient's RTSTRUCT series in notebook 11's TCIA reply
rt = SERIES_NEW[LUNG1_COLLECTION]
rt = rt[rt.Modality == 'RTSTRUCT'].sort_values(['PatientID', 'SeriesNumber', 'SeriesInstanceUID'])
RT_OF = rt.drop_duplicates('PatientID').set_index('PatientID').SeriesInstanceUID.to_dict()
SCANS['rt_uid'] = [RT_OF.get(p) if c == LUNG1_COLLECTION else None for c, p in zip(SCANS.collection, SCANS.PatientID)]
if TEST.get('patients'):
    SCANS = SCANS[SCANS.key.isin(TEST['patients'])].reset_index(drop=True)
print(SCANS.groupby('dataset').size().to_string())
print(f'{len(SCANS)} scans ({SCANS.tcia_file_size.astype(float).sum() / 1e9:.0f} GB of DICOM); without a header record: '
      f'{len(NO_HEADER)}; NSCLC-Radiomics with a contour series: {int(SCANS.rt_uid.notna().sum())}')

In [ ]:
# 8b. Masks, inputs, habitus measures and harmonized volumes (plan 7o; label-free)
from matplotlib.path import Path as MplPath

def dilate_mm(mask, sp, mm):
    """Euclidean dilation by mm, computed inside the bounding box plus a margin (as notebook 08's body mask)."""
    if not mask.any():
        return mask.copy()
    pad = [int(np.ceil(mm / s)) + 1 for s in sp]
    idx = np.where(mask)
    lo = [max(0, int(i.min()) - p) for i, p in zip(idx, pad)]
    hi = [min(n_, int(i.max()) + p + 1) for i, n_, p in zip(idx, mask.shape, pad)]
    box = tuple(slice(a, b) for a, b in zip(lo, hi))
    out = np.zeros_like(mask)
    out[box] = ndimage.distance_transform_edt(~mask[box], sampling=sp) <= mm
    return out

def largest_component(mask):
    lab_, n = ndimage.label(mask, structure=np.ones((3, 3, 3), bool))
    if n == 0:
        return None
    counts = np.bincount(lab_.ravel()); counts[0] = 0
    return lab_ == counts.argmax()

def rasterize_roi(rs_path, keep, geo, shape, pattern=GTV_PATTERN):
    """The contour of the ROI whose name matches pattern, filled on the CT slice whose recorded DICOM position is nearest
    to each contour (so scans with gaps between slices are handled), in that slice's own row and column frame."""
    ds = pydicom.dcmread(rs_path)
    names = {int(r.ROINumber): str(r.ROIName) for r in ds.StructureSetROISequence}
    num = next((n for n, nm in names.items() if re.match(pattern, nm, re.I)), None)
    if num is None:
        return None, sorted(names.values())
    ipp = np.array([[float(v) for v in d.ImagePositionPatient] for _, d in keep])
    proj = ipp @ geo['normal']
    ps = [float(v) for v in keep[0][1].PixelSpacing]                     # [between rows, between columns]
    mask = np.zeros(shape, bool)
    yy, xx = np.mgrid[0:shape[1], 0:shape[2]]
    grid = np.c_[xx.ravel(), yy.ravel()]
    for rc in ds.ROIContourSequence:
        if int(rc.ReferencedROINumber) != num:
            continue
        for c in getattr(rc, 'ContourSequence', []):
            pts = np.asarray(c.ContourData, float).reshape(-1, 3)
            if len(pts) < 3:
                continue
            k = int(np.argmin(np.abs(proj - float((pts @ geo['normal']).mean()))))
            if abs(proj[k] - float((pts @ geo['normal']).mean())) > max(0.5, abs(geo['z']) / 2 + 0.1):
                continue                                                  # no CT slice at this contour
            d = pts - ipp[k]
            col, row = d @ geo['row_cos'] / ps[1], d @ geo['col_cos'] / ps[0]
            inside = MplPath(np.c_[col, row]).contains_points(grid).reshape(shape[1:])
            mask[k] ^= inside                                             # XOR: holes are cut out
    return mask, None

def find_rtstruct(folder):
    for p in sorted(folder.rglob('*')):
        if p.is_file():
            try:
                d = pydicom.dcmread(p, stop_before_pixels=True)
            except Exception:
                continue                                                  # e.g. the LICENSE file of a TCIA download
            if getattr(d, 'Modality', '') == 'RTSTRUCT':
                return p
    raise RuntimeError('no RTSTRUCT file in the download')

def dice(a, b):
    s = a.sum() + b.sum()
    return float(2 * (a & b).sum() / s) if s else np.nan

def habitus_measures(arr, sp, core, lungs, coverage_mm):
    """Plan 7o, G2: seven label-free measures of body habitus and positioning."""
    la = lungs.sum(axis=(1, 2))
    z0 = int(np.argmax(la)) if la.any() else int(np.argmax(core.sum(axis=(1, 2))))
    sl = core[z0]
    lab2, n2 = ndimage.label(sl)
    if n2 == 0:
        return {m: np.nan for m in HABITUS}
    areas = np.bincount(lab2.ravel()) * sp[1] * sp[2] / 100.0; areas[0] = 0
    trunk = lab2 == int(np.argmax(areas))
    arms = int(((areas >= ARM_MIN_CM2).sum()) >= 2)
    rr, cc = np.where(trunk)
    ap = (rr.max() - rr.min() + 1) * sp[1]; lr = (cc.max() - cc.min() + 1) * sp[2]
    depth = ndimage.distance_transform_edt(trunk, sampling=sp[1:])
    fat = trunk & (depth <= FAT_RING_MM) & (arr[z0] >= FAT_HU[0]) & (arr[z0] <= FAT_HU[1])
    rows_any = np.where(core.any(axis=(0, 2)))[0]
    table = (core.shape[1] - 1 - rows_any.max()) * sp[1] if len(rows_any) else np.nan
    cy, cx = ndimage.center_of_mass(trunk)
    off = float(np.hypot((cy - (core.shape[1] - 1) / 2) * sp[1], (cx - (core.shape[2] - 1) / 2) * sp[2]))
    return {'ap_lr_ratio': float(ap / lr), 'subcut_fat_cm2': float(fat.sum() * sp[1] * sp[2] / 100.0),
            'arms_beside_trunk': arms, 'table_height_mm': float(table), 'centroid_offset_mm': off,
            'scan_length_mm': float(coverage_mm), 'lung_volume_l': float(lungs.sum() * np.prod(sp) / 1e6),
            'habitus_slice': z0}

def harmonize(img, arrays, lungs, core):
    """Plan 7o, G3: 1 x 1 mm in-plane, 5-mm slabs, a fixed 360 x 360 mm field around the body centroid, and the z-range
    from 20 mm above the lung apex to 20 mm below the lung base. Returns {name: (z, y, x) array at 1 x 1 x 5 mm}."""
    spx, spy, spz = img.GetSpacing()
    ref = lungs if lungs.any() else core
    zs = np.where(ref.any(axis=(1, 2)))[0]
    z_lo = zs.min() - HARM_Z_MARGIN_MM / spz; z_hi = zs.max() + HARM_Z_MARGIN_MM / spz
    cz = np.where(core[zs.min():zs.max() + 1].any(axis=0))
    cy, cx = (np.mean(cz[0]), np.mean(cz[1])) if len(cz[0]) else ((core.shape[1] - 1) / 2, (core.shape[2] - 1) / 2)
    half = HARM_FOV_MM / 2
    origin = img.TransformContinuousIndexToPhysicalPoint((cx - half / spx, cy - half / spy, float(z_lo)))
    n_xy = int(round(HARM_FOV_MM / HARM_INPLANE_MM))
    n_z = int(np.floor((z_hi - z_lo) * spz / HARM_SLAB_MM)) * int(HARM_SLAB_MM)
    out = {}
    for name, a in arrays.items():
        src = like(a, img, np.int16)
        r = sitk.Resample(src, [n_xy, n_xy, max(n_z, int(HARM_SLAB_MM))], sitk.Transform(), sitk.sitkLinear, origin,
                          (HARM_INPLANE_MM, HARM_INPLANE_MM, 1.0), img.GetDirection(), float(AIR_HU), sitk.sitkFloat32)
        v = sitk.GetArrayFromImage(r)                                  # (z at 1 mm, y, x)
        k = int(HARM_SLAB_MM)
        v = v[:(v.shape[0] // k) * k].reshape(-1, k, n_xy, n_xy).mean(axis=1)
        out[name] = v
    geo = sitk.Image([n_xy, n_xy, 1], sitk.sitkInt16)
    geo.SetOrigin(origin); geo.SetSpacing((HARM_INPLANE_MM, HARM_INPLANE_MM, HARM_SLAB_MM)); geo.SetDirection(img.GetDirection())
    return out, geo

def harm_image(v, geo):
    out = sitk.GetImageFromArray(np.clip(np.round(v), -32768, 32767).astype(np.int16))
    out.SetOrigin(geo.GetOrigin()); out.SetSpacing(geo.GetSpacing()); out.SetDirection(geo.GetDirection())
    return out

def cnn_array(v):
    """2 x 2 mm in-plane (block mean) and a fixed 64-slab depth centred on the volume (padded with air)."""
    k = int(round(CNN_INPLANE_MM / HARM_INPLANE_MM))
    z, y, x = v.shape
    v = v[:, :(y // k) * k, :(x // k) * k].reshape(z, y // k, k, x // k, k).mean(axis=(2, 4))
    nz = CNN_SHAPE[0]
    if z >= nz:
        s = (z - nz) // 2; v = v[s:s + nz]
    else:
        a = (nz - z) // 2; v = np.pad(v, ((a, nz - z - a), (0, 0), (0, 0)), constant_values=AIR_HU)
    return np.clip(np.round(v), -32768, 32767).astype(np.int16)

def run_ts(inp, out_dir, args, attempts=2):
    for k in range(attempts):                                        # one retry (e.g. a weight download that broke off)
        try:
            return _run_ts(inp, out_dir, args)
        except Exception:
            if k == attempts - 1:
                raise
            shutil.rmtree(out_dir, ignore_errors=True)
            time.sleep(30)

def _run_ts(inp, out_dir, args):
    """TotalSegmentator with its output in a log file, not a pipe (helper processes it starts can keep a pipe open after
    it has finished), a time limit that ends the whole process tree, and a check that the expected masks exist."""
    out_dir.mkdir(parents=True, exist_ok=True)
    log = out_dir.parent / f'{out_dir.name}_ts.log'
    with open(log, 'wb') as fh:
        p = subprocess.Popen([str(TS_BIN), '-i', str(inp), '-o', str(out_dir), '--quiet', *args], env=TS_ENVIRON,
                             stdout=fh, stderr=subprocess.STDOUT, stdin=subprocess.DEVNULL,
                             start_new_session=(os.name != 'nt'))
        try:
            rc = p.wait(timeout=TS_TIMEOUT)
        except subprocess.TimeoutExpired:
            if os.name == 'nt':
                subprocess.run(['taskkill', '/F', '/T', '/PID', str(p.pid)], capture_output=True)
            else:
                import signal
                os.killpg(p.pid, signal.SIGKILL)
            p.wait()
            raise RuntimeError(f'TotalSegmentator {args[:2]} took longer than {TS_TIMEOUT} s and was stopped')
    if rc != 0:
        raise RuntimeError(f'TotalSegmentator {args[:2]} (exit {rc}): ' + log.read_text(errors='replace')[-600:])
    want = [f'{r}.nii.gz' for r in LUNG_ROIS] if '--roi_subset' in args else ['lung_nodules.nii.gz']
    missing = [w for w in want if not (out_dir / w).exists()]
    if missing:
        raise RuntimeError(f'TotalSegmentator {args[:2]} finished without {missing}')

def read_mask(path, ref_shape):
    a = sitk.GetArrayFromImage(sitk.ReadImage(str(path))) > 0
    assert a.shape == ref_shape, f'mask shape {a.shape} differs from the image {ref_shape}'
    return a

def work12(key):
    d = WORK / 'nifti'
    names = ('original', 'core', 'dil', 'gtv') + VARIANTS12[1:] + HARM_VARIANTS
    return {n: d / f'{key}__{n}.nii.gz' for n in names}

def record12_path(key):
    return CACHE12 / 'records' / f'{key}.json'

def record12(key):
    p = record12_path(key)
    return json.loads(p.read_text()) if p.exists() else {}

def emb12_path(key):
    return CACHE12 / 'embeddings' / f'{key}.npz'

def emb12_ok(key):
    p, r = emb12_path(key), record12(key)
    if not p.exists() or r.get('stage') != 'segmented':
        return False
    need = [v for v in VARIANTS12 if v != 'tumour' or r.get('tumour_detected')] + list(HARM_VARIANTS)
    with np.load(p) as z:
        return json.loads(str(z['meta'])).get('emb12_key') == EMB12_KEY and all(f'{e}__{v}' in z.files
                                                                                  for e in ENCODERS for v in need)

In [ ]:
# 8c. Stage A (CPU, parallel): download, rebuild the body mask (checked against the saved one), contours
def prepare12(key, uid, ref_fraction, ref_area, rt_uid):
    paths = work12(key)
    r0 = record12(key)
    if (r0.get('prep12_key') == PREP12_KEY and r0.get('stage') in ('prepared', 'segmented')
            and paths['original'].exists()):
        return {'key': key, 'status': 'ready'}      # otherwise (new, or the work disk was lost): prepare (again)
    t0 = time.time()
    tmp, zpath = WORK / 'dicom' / key, WORK / 'dicom' / f'{key}.zip'
    try:
        shutil.rmtree(tmp, ignore_errors=True)
        download_series(uid, zpath)
        with zipfile.ZipFile(zpath) as z:
            z.extractall(tmp)
        keep, geo = read_series(tmp, uid)
        img = to_image(keep, geo)
        arr = sitk.GetArrayFromImage(img)
        sp = img.GetSpacing()[::-1]
        core, dil = body_mask(arr, sp)
        area = core.sum(axis=(1, 2)) * sp[1] * sp[2] / 100.0
        frac = float(dil.mean()); a_mean = float(area[area > 0].mean()) if (area > 0).any() else np.nan
        repro = bool(abs(frac - ref_fraction) <= REPRO_BODY_TOL and abs(a_mean - ref_area) <= 1e-6 * max(1.0, ref_area))
        sitk.WriteImage(like(arr, img, np.int16), str(paths['original']), useCompression=True)
        sitk.WriteImage(like(core, img, np.uint8), str(paths['core']), useCompression=True)
        sitk.WriteImage(like(dil, img, np.uint8), str(paths['dil']), useCompression=True)
        gtv_info = {}
        if rt_uid:
            rz, rdir = WORK / 'dicom' / f'{key}__rt.zip', WORK / 'dicom' / f'{key}__rt'
            download_series(rt_uid, rz)
            with zipfile.ZipFile(rz) as z:
                z.extractall(rdir)
            g, names = rasterize_roi(find_rtstruct(rdir), keep, geo, arr.shape)
            if g is not None:
                sitk.WriteImage(like(g, img, np.uint8), str(paths['gtv']), useCompression=True)
            gtv_info = {'gtv_found': g is not None, 'gtv_voxels': int(g.sum()) if g is not None else 0,
                        'roi_names_if_missing': names}
            rz.unlink(missing_ok=True); shutil.rmtree(rdir, ignore_errors=True)
        rec = {'key': key, 'SeriesInstanceUID': uid, 'prep12_key': PREP12_KEY, 'stage': 'prepared',
               'body_fraction_dilated': frac, 'body_area_cm2': a_mean, 'body_reproduced': repro,
               'coverage_mm': geo['coverage_mm'], 'spacing_zyx': list(sp), 'shape_zyx': list(arr.shape),
               'nonuniform_spacing': bool(geo['nonuniform_spacing']),
               **gtv_info, 'prep_seconds': round(time.time() - t0, 1)}
        write_json_atomic(record12_path(key), rec)
        return {'key': key, 'status': 'ok'}
    except Exception as e:
        return {'key': key, 'status': 'failed', 'error': repr(e)[:400]}
    finally:
        shutil.rmtree(tmp, ignore_errors=True)
        zpath.unlink(missing_ok=True)

In [ ]:
# 8d. Stage B (GPU, one scan at a time): lungs and tumour, the new inputs, habitus, harmonized and CNN volumes
def segment12(key):
    paths, rec = work12(key), record12(key)
    t0 = time.time()
    try:
        seg = WORK / 'seg' / key
        run_ts(paths['original'], seg / 'lungs', ['--fast', '--roi_subset', *LUNG_ROIS])
        run_ts(paths['original'], seg / 'nodules', ['-ta', 'lung_nodules'])
        img = sitk.ReadImage(str(paths['original']))
        arr = sitk.GetArrayFromImage(img).astype(np.int16)
        sp = img.GetSpacing()[::-1]
        core = read_mask(paths['core'], arr.shape); dil = read_mask(paths['dil'], arr.shape)
        lungs = np.zeros(arr.shape, bool)
        for roi in LUNG_ROIS:
            lungs |= read_mask(seg / 'lungs' / f'{roi}.nii.gz', arr.shape)
        nod = read_mask(seg / 'nodules' / 'lung_nodules.nii.gz', arr.shape)
        tumour = largest_component(nod)
        lungs_only = np.where(dilate_mm(lungs, sp, LUNG_DILATE_MM), arr, AIR_HU)
        sitk.WriteImage(like(lungs_only, img), str(paths['lungs']), useCompression=True)
        info = {'lung_voxels': int(lungs.sum()), 'tumour_detected': tumour is not None,
                'n_nodule_components': int(ndimage.label(nod, structure=np.ones((3, 3, 3), bool))[1]),
                'tumour_volume_ml': float(tumour.sum() * np.prod(sp) / 1000) if tumour is not None else 0.0}
        if tumour is not None:
            t_only = np.where(dilate_mm(tumour, sp, TUMOUR_DILATE_MM), arr, AIR_HU)
            sitk.WriteImage(like(t_only, img), str(paths['tumour']), useCompression=True)
        if paths['gtv'].exists():
            gtv = read_mask(paths['gtv'], arr.shape)
            info['gtv_dice'] = dice(tumour, gtv) if tumour is not None else 0.0
            info['gtv_dice_all_nodules'] = dice(nod, gtv)
        hab = habitus_measures(arr, sp, core, lungs, rec.get('coverage_mm', np.nan))
        bg = arr.copy(); bg[dil] = AIR_HU
        harm, geo = harmonize(img, {'harm_full': arr, 'harm_background': bg}, lungs, core)
        for v in HARM_VARIANTS:
            sitk.WriteImage(harm_image(harm[v], geo), str(paths[v]), useCompression=True)
        tmpc = CACHE12 / 'cnn' / f'{key}.tmp.npz'
        np.savez_compressed(tmpc, full=cnn_array(harm['harm_full']), background=cnn_array(harm['harm_background']))
        tmpc.replace(CACHE12 / 'cnn' / f'{key}.npz')
        if key in QC12_KEYS:
            qc12_picture(key, arr, lungs, tumour, harm, CACHE12 / 'qc' / f'{key}.png')
        rec.update({'stage': 'segmented', **info, **hab, 'harm_slabs': int(harm['harm_full'].shape[0]),
                    'seg_seconds': round(time.time() - t0, 1)})
        write_json_atomic(record12_path(key), rec)
        return {'key': key, 'status': 'ok'}
    except Exception as e:
        return {'key': key, 'status': 'failed', 'error': repr(e)[:400]}
    finally:
        shutil.rmtree(WORK / 'seg' / key, ignore_errors=True)

def qc12_picture(key, arr, lungs, tumour, harm, path):
    from matplotlib.figure import Figure
    z = int(np.argmax(lungs.sum(axis=(1, 2)))) if lungs.any() else arr.shape[0] // 2
    fig = Figure(figsize=(12, 4.2)); ax = fig.subplots(1, 3); kw = dict(cmap='gray', vmin=-1000, vmax=400)
    ax[0].imshow(arr[z], **kw); ax[0].contour(lungs[z], levels=[0.5], colors='c', linewidths=0.6)
    if tumour is not None:
        zt = int(np.argmax(tumour.sum(axis=(1, 2))))
        ax[1].imshow(arr[zt], **kw); ax[1].contour(tumour[zt], levels=[0.5], colors='r', linewidths=0.8)
        ax[1].set_title('tumour (automatic)')
    else:
        ax[1].set_title('no tumour detected')
    hz = harm['harm_full'].shape[0] // 2
    ax[2].imshow(harm['harm_full'][hz], **kw); ax[2].set_title('harmonized (1 x 1 x 5 mm)')
    ax[0].set_title('lungs (automatic)')
    for a in ax:
        a.axis('off')
    fig.suptitle(key, fontsize=9); fig.tight_layout(); fig.savefig(path, dpi=80)

In [ ]:
# 9. Part 1: prepare, segment and embed (label-free; resumable)
QC12_KEYS = set(SCANS.sort_values('key').groupby('dataset').key.head(N_QC))

def fake_part1():
    """Local code tests only: random records and embeddings instead of images (seeded, label-free)."""
    rng = np.random.default_rng(12)
    for r in SCANS.itertuples():
        if not emb12_ok(r.key):
            det = bool(rng.random() < 0.9)
            rec = {'key': r.key, 'prep12_key': PREP12_KEY, 'stage': 'segmented', 'fake': True, 'body_reproduced': True,
                   'tumour_detected': det, 'ap_lr_ratio': rng.uniform(.5, .9), 'subcut_fat_cm2': rng.uniform(10, 200),
                   'arms_beside_trunk': int(rng.random() < .3), 'table_height_mm': rng.uniform(20, 150),
                   'centroid_offset_mm': rng.uniform(0, 40), 'scan_length_mm': rng.uniform(250, 600),
                   'lung_volume_l': rng.uniform(2, 7)}
            if r.rt_uid:
                rec['gtv_dice'] = rng.uniform(0, .9)
            write_json_atomic(record12_path(r.key), rec)
            arrays = {}
            for v in ('original', 'lungs') + (('tumour',) if det else ()) + HARM_VARIANTS:
                arrays[f'ctfm__{v}'] = rng.normal(size=512).astype(np.float32)
                arrays[f'merlin__{v}'] = np.abs(rng.normal(size=2048)).astype(np.float32)
            np.savez(emb12_path(r.key), meta=json.dumps({'emb12_key': EMB12_KEY, 'fake': True}), **arrays)

t_part1 = time.time()
failed12 = []
if TEST.get('fake_images'):
    fake_part1()
else:
    MODELS = None
    todo = [r for r in SCANS.itertuples() if not emb12_ok(r.key)]
    print(f'Scans to finish: {len(todo)} of {len(SCANS)} (the others are finished on Drive)')
    # Stage A and B are interleaved in batches, so that the work disk never holds more than one batch
    BATCH = 48
    todo.sort(key=lambda r: float(r.tcia_file_size or 0))              # the smallest scan first: a quick full check
    batches = [todo[:1]] + [todo[i:i + BATCH] for i in range(1, len(todo), BATCH)] if todo else []
    for bi, batch in enumerate(batches):
        b0 = bi
        resA = Parallel(n_jobs=PREP_JOBS)(delayed(prepare12)(r.key, r.SeriesInstanceUID, float(r.ref_body_fraction),
                                                             float(r.ref_body_area), r.rt_uid)
                                          for r in batch)
        failed12 += [x for x in resA if x['status'] == 'failed']
        for r in batch:
            rc = record12(r.key)
            if rc.get('stage') == 'prepared' and work12(r.key)['original'].exists():
                x = segment12(r.key)
                if x['status'] == 'failed':
                    failed12.append(x)
        if MODELS is None:
            MODELS = {'ctfm': load_ctfm().to(DEVICE), 'merlin': load_merlin().to(DEVICE)}
        items, items_h = [], []
        for r in batch:
            rc, p = record12(r.key), work12(r.key)
            if rc.get('stage') != 'segmented' or emb12_ok(r.key) or not p['original'].exists():
                continue
            vs = [v for v in VARIANTS12 if v != 'tumour' or rc.get('tumour_detected')]
            items.append((r.key, {v: p[v] for v in vs}, 'original'))
        tmp_dir = WORK / 'emb_tmp'; tmp_dir.mkdir(exist_ok=True)
        f1 = embed_items(items, lambda k: tmp_dir / f'{k}__a.npz', MODELS, label='original + lungs + tumour')
        items_h = [(k, {v: work12(k)[v] for v in HARM_VARIANTS}, 'harm_full') for k, _, _ in items]
        f2 = embed_items(items_h, lambda k: tmp_dir / f'{k}__h.npz', MODELS, label='harmonized')
        failed12 += [{'key': f['key'], 'status': 'failed', 'error': f['error']} for f in f1 + f2]
        for k, _, _ in items:
            pa, ph = tmp_dir / f'{k}__a.npz', tmp_dir / f'{k}__h.npz'
            if not (pa.exists() and ph.exists()):
                continue
            arrays, cos = {}, {}
            with np.load(pa) as za, np.load(ph) as zh:
                for z in (za, zh):
                    arrays.update({n: z[n] for n in z.files if n != 'meta'})
            with np.load(emb_file(k, 'original')) as zo:    # reproduction: the original as embedded before
                for e in ENCODERS:
                    a, b = arrays[f'{e}__original'], zo[f'{e}__original']
                    cos[e] = float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
            meta = {'emb12_key': EMB12_KEY, 'original_cosine_to_earlier': cos}
            tmp = emb12_path(k).with_name(f'{k}.tmp.npz')
            np.savez(tmp, meta=json.dumps(meta), **arrays); tmp.replace(emb12_path(k))
            rc = record12(k); rc['original_cosine_to_earlier'] = cos; write_json_atomic(record12_path(k), rc)
            pa.unlink(); ph.unlink()
        for r in batch:                                     # free the work disk
            if emb12_ok(r.key):
                for p in work12(r.key).values():
                    p.unlink(missing_ok=True)
        bad = [r.key for r in batch if not emb12_ok(r.key)]
        if bi == 0 and bad:
            raise RuntimeError('The first (check) scan did not finish; nothing else was started. Errors: '
                               + '; '.join(f"{f['key']}: {f.get('error')}" for f in failed12)[-1500:])
        if len(batch) >= 8 and len(bad) == len(batch):
            raise RuntimeError('Every scan of this batch failed, so the problem is not one scan; stopping. Errors: '
                               + '; '.join(f"{f['key']}: {f.get('error')}" for f in failed12[-5:])[-1500:])
        done = sum(emb12_ok(r.key) for r in SCANS.itertuples())
        print(f'  batch {bi + 1}/{len(batches)}: {done}/{len(SCANS)} scans finished ({(time.time() - t_part1) / 3600:.1f} h, '
              f'{shutil.disk_usage(WORK).free / 1e9:.0f} GB free)')
print(f'Part 1 finished in {(time.time() - t_part1) / 60:.1f} min')
for f in failed12:
    print(f"  FAILED {f['key']}: {f.get('error')}")

REC = pd.DataFrame([{**{'key': k}, **record12(k)} for k in SCANS.key])
REC['embedded'] = [emb12_ok(k) for k in SCANS.key]
REC = SCANS[['dataset', 'key']].merge(REC, on='key', how='left')
REC.drop(columns=[c for c in REC if c.startswith('roi_names')], errors='ignore').to_csv(CACHE12 / 'scan_records.csv', index=False)
pd.DataFrame(failed12, columns=['key', 'status', 'error']).to_csv(CACHE12 / 'failures_this_session.csv', index=False)
cov12 = REC.groupby('dataset').agg(scans=('key', 'size'), embedded=('embedded', 'sum'),
                                   body_reproduced=('body_reproduced', lambda s: int(s.fillna(False).astype(bool).sum())),
                                   tumour_detected=('tumour_detected', lambda s: int(s.fillna(False).astype(bool).sum()))).reset_index()
cov12['coverage'] = cov12.embedded / cov12.scans
show(cov12)
# segmentation check against the manual GTV-1 of NSCLC-Radiomics (plan 7o): share with Dice > 0.1, median Dice
g = REC[REC.dataset == 'NSCLC-Radiomics']
gd = g.gtv_dice.astype(float) if 'gtv_dice' in g else pd.Series(dtype=float)
TUMOUR_CHECK = {'patients_with_gtv': int(gd.notna().sum()),
                'share_dice_above': float((gd > TUMOUR_CHECK_DICE).mean()) if gd.notna().any() else np.nan,
                'median_dice': float(gd.median()) if gd.notna().any() else np.nan}
TUMOUR_CHECK['segmentation_reliable'] = bool(TUMOUR_CHECK['share_dice_above'] >= TUMOUR_CHECK_MIN_SHARE)
print('Tumour segmentation check (NSCLC-Radiomics GTV-1):', TUMOUR_CHECK)
if 'original_cosine_to_earlier' in REC and REC.original_cosine_to_earlier.notna().any():
    cs = pd.DataFrame(REC.original_cosine_to_earlier.dropna().tolist())
    print('Original embeddings against the earlier notebooks: median cosine',
          {e: round(float(cs[e].median()), 6) for e in cs})
if TEST.get('stop_after_images'):
    raise SystemExit('Local test: stopping after the image part.')

## Part 2 · The experiments of plan 7o (once on shuffled labels as a rehearsal, then on the real labels)

In [ ]:
# 10. Model building blocks (notebook 11, unchanged)
class TokenBinarizer(BaseEstimator, TransformerMixin):
    """One 0/1 column per word used by at least `min_count` training patients."""
    def __init__(self, min_count=MIN_COUNT):
        self.min_count = min_count
    def _docs(self, X):
        X = X.iloc[:, 0] if isinstance(X, pd.DataFrame) else pd.Series(np.asarray(X).ravel())
        return X.fillna('').astype(str).tolist()
    def fit(self, X, y=None):
        counts = {}
        for d in self._docs(X):
            for t in set(d.split()):
                counts[t] = counts.get(t, 0) + 1
        self.vocab_ = {t: i for i, t in enumerate(sorted(t for t, c in counts.items() if c >= self.min_count))}
        return self
    def transform(self, X):
        rows, cols = [], []
        docs = self._docs(X)
        for i, d in enumerate(docs):
            for t in set(d.split()):
                j = self.vocab_.get(t)
                if j is not None:
                    rows.append(i)
                    cols.append(j)
        return sparse.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(len(docs), max(len(self.vocab_), 1)))

def column_transformer(cols):
    parts = []
    for c in cols:
        if c in CATEGORICAL:
            parts.append((c, Pipeline([('fill', SimpleImputer(strategy='constant', fill_value='missing',
                                                              keep_empty_features=True)),
                                       ('onehot', OneHotEncoder(handle_unknown='infrequent_if_exist',
                                                                min_frequency=MIN_COUNT, sparse_output=True))]), [c]))
        elif c in TEXT:
            parts.append((c, TokenBinarizer(), [c]))
        else:
            steps = [('log', FunctionTransformer(np.log1p, feature_names_out='one-to-one'))] if c in LOG_NUMERIC else []
            steps += [('fill', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
                      ('scale', StandardScaler())]
            parts.append((c, Pipeline(steps), [c]))
    return ColumnTransformer(parts, remainder='drop', sparse_threshold=0.3)

def auc_binary(y, s):
    """AUC by ranks (ties count one half); y in {0,1}."""
    n1 = int(y.sum()); n0 = len(y) - n1
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return (r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)

def auc(y, P):
    k = P.shape[1]
    if k == 2:
        return auc_binary(y, P[:, 1])
    return float(np.nanmean([auc_binary((y == j).astype(int), P[:, j]) for j in range(k)]))

def repeated_splits(y, seed=SEED):
    return [list(StratifiedKFold(N_SPLITS, shuffle=True, random_state=seed + r).split(np.zeros(len(y)), y))
            for r in range(N_REPEATS)]

def mean_auc(y, P):
    return float(np.mean([auc(y, P[r]) for r in range(P.shape[0])]))

def boot_ci(y, P):
    """Percentile 95% CI of the repeat-averaged AUC over patient bootstrap resamples."""
    rng = np.random.default_rng(SEED)
    k = P.shape[2]
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < k:
            continue
        vals.append(np.mean([auc(y[idx], P[r][idx]) for r in range(P.shape[0])]))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_diff(y, PA, PB):
    """Repeat-averaged AUC(A) - AUC(B) on the same patients, with a paired bootstrap 95% CI."""
    rng = np.random.default_rng(SEED)
    vals = []
    while len(vals) < N_BOOT:
        idx = rng.choice(len(y), len(y), replace=True)
        if len(np.unique(y[idx])) < 2:
            continue
        vals.append(np.mean([auc(y[idx], PA[r][idx]) - auc(y[idx], PB[r][idx]) for r in range(PA.shape[0])]))
    return mean_auc(y, PA) - mean_auc(y, PB), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def boot_mean(x):
    rng = np.random.default_rng(SEED)
    x = np.asarray(x, float)
    vals = [x[rng.choice(len(x), len(x), replace=True)].mean() for _ in range(N_BOOT)]
    return float(x.mean()), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

# --- metadata models (notebook 03) ---
def _fit_fold_df(df, y, cols, tr, te, k):
    m = Pipeline([('prep', column_transformer(cols)), ('model', LogisticRegression(**LR_PARAMS))]).fit(df.iloc[tr], y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(df.iloc[te])
    return te, proba

def cv_oof_df(df, y, cols, splits):
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_df)(df, y, cols, tr, te, k) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    for (r, _, _), (te, proba) in zip(jobs, res):
        P[r, te] = proba
    return P

# --- the same logistic regression on embeddings (standardized inside each training fold) ---
def _fit_fold_matrix(X, y, tr, te, k, extras):
    m = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(**LR_PARAMS))]).fit(X[tr], y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(X[te])
    out = {}
    for name, Xe in extras.items():           # the fitted fold model applied to other inputs of the same patients
        pe = np.full((len(te), k), np.nan)
        ok = np.where(~np.isnan(Xe[te]).any(axis=1))[0]
        if len(ok):
            tmp = np.zeros((len(ok), k))
            tmp[:, m.classes_] = m.predict_proba(Xe[te][ok])
            pe[ok] = tmp
        out[name] = pe
    return te, proba, out

def cv_oof_matrix(X, y, splits, extras=None):
    extras = extras or {}
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_matrix)(X, y, tr, te, k, extras) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    E = {n: np.full((len(splits), len(y), k), np.nan) for n in extras}
    for (r, _, _), (te, proba, out) in zip(jobs, res):
        P[r, te] = proba
        for n, pe in out.items():
            E[n][r, te] = pe
    return P, E

def _fit_fold_combined(df, E, y, cols, tr, te, k):
    ct = column_transformer(cols).fit(df.iloc[tr])
    sc = StandardScaler().fit(E[tr])
    Xtr = sparse.hstack([sparse.csr_matrix(ct.transform(df.iloc[tr])), sparse.csr_matrix(sc.transform(E[tr]))]).tocsr()
    Xte = sparse.hstack([sparse.csr_matrix(ct.transform(df.iloc[te])), sparse.csr_matrix(sc.transform(E[te]))]).tocsr()
    m = LogisticRegression(**LR_PARAMS).fit(Xtr, y[tr])
    proba = np.zeros((len(te), k))
    proba[:, m.classes_] = m.predict_proba(Xte)
    return te, proba

def cv_oof_combined(df, E, y, cols, splits):
    k = int(y.max()) + 1
    jobs = [(r, tr, te) for r, folds in enumerate(splits) for tr, te in folds]
    res = Parallel(n_jobs=N_JOBS)(delayed(_fit_fold_combined)(df, E, y, cols, tr, te, k) for _, tr, te in jobs)
    P = np.zeros((len(splits), len(y), k))
    for (r, _, _), (te, proba) in zip(jobs, res):
        P[r, te] = proba
    return P

# --- exact permutation tests on one fixed label-independent split (notebook 03, 7b.6) ---
def encoded_folds_df(df, cols):
    out = []
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(np.zeros(len(df))):
        ct = column_transformer(cols).fit(df.iloc[tr])
        out.append((tr, te, ct.transform(df.iloc[tr]), ct.transform(df.iloc[te])))
    return out

def scaled_folds(X):
    out = []
    for tr, te in KFold(N_SPLITS, shuffle=True, random_state=SEED).split(X):
        sc = StandardScaler().fit(X[tr])
        out.append((tr, te, sc.transform(X[tr]), sc.transform(X[te])))
    return out

def _single_cv_proba(folds, y):
    k = int(y.max()) + 1
    P = np.zeros((len(y), k))
    for tr, te, Xtr, Xte in folds:
        m = LogisticRegression(**LR_PARAMS).fit(Xtr, y[tr])
        P[np.ix_(te, m.classes_)] = m.predict_proba(Xte)
    return P

def _perm_chunk(folds, ys):
    return [auc(yp, _single_cv_proba(folds, yp)) for yp in ys]

def permutation_test(folds, y):
    observed = auc(y, _single_cv_proba(folds, y))
    rng = np.random.default_rng(SEED)
    perms = [rng.permutation(y) for _ in range(N_PERM)]
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_perm_chunk)(folds, c) for c in chunks))
    return observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), null

# --- within-group AUC and its exact test (notebook 04, part D) ---
def within_site_auc(y, s, site):
    num, den, n_sites = 0.0, 0, 0
    for st in np.unique(site):
        m = site == st
        yy, ss = y[m], s[m]
        n1 = int(yy.sum()); n0 = len(yy) - n1
        if n1 and n0:
            r = rankdata(ss)
            num += r[yy == 1].sum() - n1 * (n1 + 1) / 2
            den += n1 * n0
            n_sites += 1
    return (num / den if den else np.nan), den, n_sites

def within_site_estimate(y, P, site):
    vals = [within_site_auc(y, P[r][:, 1], site)[0] for r in range(P.shape[0])]
    _, pairs, n_sites = within_site_auc(y, P[0][:, 1], site)
    if pairs == 0:
        return {'auc': np.nan, 'ci_low': np.nan, 'ci_high': np.nan, 'n_pairs': 0, 'n_sites_with_pairs': 0}
    rng = np.random.default_rng(SEED)
    boots = []
    groups = [np.where(site == st)[0] for st in np.unique(site)]
    while len(boots) < N_BOOT:
        idx = np.concatenate([rng.choice(g, len(g), replace=True) for g in groups])
        v = np.mean([within_site_auc(y[idx], P[r][idx, 1], site[idx])[0] for r in range(P.shape[0])])
        if not np.isnan(v):
            boots.append(v)
    return {'auc': float(np.mean(vals)), 'ci_low': float(np.percentile(boots, 2.5)),
            'ci_high': float(np.percentile(boots, 97.5)), 'n_pairs': int(pairs), 'n_sites_with_pairs': n_sites}

def _strata_chunk(folds, ys, strata, keep):
    return [within_site_auc(yp[keep], _single_cv_proba(folds, yp)[keep, 1], strata[keep])[0] for yp in ys]

def within_group_permutation_test(folds, y, strata, keep):
    observed = within_site_auc(y[keep], _single_cv_proba(folds, y)[keep, 1], strata[keep])[0]
    rng = np.random.default_rng(SEED)
    groups = [np.where(strata == s)[0] for s in np.unique(strata)]
    perms = []
    for _ in range(N_PERM):
        yp = y.copy()
        for g in groups:
            yp[g] = y[rng.permutation(g)]
        perms.append(yp)
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_strata_chunk)(folds, c, strata, keep) for c in chunks))
    return observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), null

def bh(p):
    """Benjamini-Hochberg q-values; missing p-values stay missing and are not counted."""
    p = np.asarray(p, float)
    q = np.full_like(p, np.nan)
    ok = np.where(~np.isnan(p))[0]
    order = ok[np.argsort(p[ok])]
    prev = 1.0
    for rank, j in reversed(list(enumerate(order, 1))):
        prev = min(prev, p[j] * len(ok) / rank)
        q[j] = prev
    return q

In [ ]:
# 11. Datasets (notebook 11, unchanged)
def task_frame(tid, shuffle_seed=None):
    """Task patients with metadata, groups, headers and body size. shuffle_seed: shuffled labels (rehearsal)."""
    df = labels[labels.task_id == tid].merge(feat, on=KEY, how='left', validate='one_to_one')
    df['key'] = df.collection + '__' + df.PatientID
    classes = sorted(df.label.unique())
    y = df.label.map({c: i for i, c in enumerate(classes)}).to_numpy()
    if shuffle_seed is not None:
        y = np.random.default_rng(shuffle_seed).permutation(y)
    df['site'] = df.PatientID.map(site_of)
    df = df.merge(polish, on=KEY, how='left')
    df['language'] = np.where(df.polish.eq(True), 'Polish', 'other')
    return df.reset_index(drop=True), y, classes

def load_embeddings(keys, variants, path_of):
    out = {(e, v): [] for e in ENCODERS for v in variants}
    for k in keys:
        p = path_of(k)
        ok = p is not None and p.exists()
        z = np.load(p) if ok else None
        for e in ENCODERS:
            for v in variants:
                name = f'{e}__{v}'
                out[(e, v)].append(z[name] if ok and name in z.files else None)
        if z is not None:
            z.close()
    res = {}
    for (e, v), lst in out.items():
        d = next((len(a) for a in lst if a is not None), 1)
        res[(e, v)] = np.vstack([a if a is not None else np.full(d, np.nan, np.float32) for a in lst]).astype(float)
    return res

def load_inputs(keys):
    """{(encoder, variant): matrix} from whichever notebook holds each scan's embedding."""
    out = {}
    for v in VARIANTS:
        out.update(load_embeddings(keys, (v,), lambda k, v=v: emb_file(k, v)))
    return out

def dataset_frame(name, shuffle_seed=None):
    """Patients with embeddings, labels in the shared class order (class 1: squamous). The source and NSCLC Radiogenomics
    use notebook 05's task frame and CPTAC lung notebook 08's frame, so their rows and shuffles are exactly those of
    notebooks 05, 07, 08 and 09; the new targets are built the same way (sorted by key)."""
    if name in SOURCES or name == 'NSCLC Radiogenomics':
        df, y, classes = task_frame(name if name in SOURCES else NSCLC_TASK, shuffle_seed)
        assert tuple(classes) == CLASSES[name], f'{name}: class order {classes} differs from {CLASSES[name]}'
    else:
        df = TARGETS[name]
        y = (df.label == CLASSES[name][1]).astype(int).to_numpy()
        if shuffle_seed is not None:
            y = np.random.default_rng(shuffle_seed).permutation(y)
    keep = df.key.isin(complete).to_numpy()
    return df[keep].reset_index(drop=True), y[keep], int((~keep).sum())

def with_headers(df):
    """The scan's header record and body size, prepared as in notebooks 05 and 06 (categorical fields as text)."""
    h = pd.DataFrame([header_of(k) for k in df.key])
    wanted = ['body_area_cm2', 'hdr_contrast_agent_given', 'hdr_coverage_mm', 'hdr_manufacturer'] + HDR_COLS
    df = df.drop(columns=[c for c in wanted if c in df]).merge(h[['key'] + [c for c in wanted if c in h]], on='key',
                                                                how='left')
    for c in wanted:
        if c not in df:
            df[c] = np.nan
    for c in HDR_CATEGORICAL + ['hdr_manufacturer']:
        df[c] = df[c].map(lambda v: str(v) if pd.notna(v) else np.nan).astype(object)
    df['manufacturer'] = df.hdr_manufacturer.str.upper().str.extract(r'^\s*([A-Z]+)', expand=False)   # as experiment C
    return df

In [ ]:
# 12. Transfer building blocks (notebook 11, unchanged)
def fit_apply(Xa, ya, Xb):
    """The probe (scaling + logistic regression, notebook 05's settings) fitted on all of A, applied to B. The score is
    the log-odds of class 1: the same ranking (and AUC) as the probability, but it cannot round to exactly 0 or 1 when
    B's embeddings lie far from A's (settled item 6)."""
    m = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(**LR_PARAMS))]).fit(Xa, ya)
    assert list(m.classes_) == [0, 1]
    return m.decision_function(Xb)

def as_P(s):
    """One score vector in the (repeats, patients, classes) shape of notebook 05's AUC and bootstrap functions
    (only the class-1 column is read, by rank)."""
    return np.stack([-s, s], axis=1)[None]

def _external_chunk(Xa, ys, Xb, yb):
    return [auc_binary(yb, fit_apply(Xa, yp, Xb)) for yp in ys]

def external_test(Xa, ya, Xb, yb):
    """External AUC; permutation test by refitting on permuted A labels (settled item 6: descriptive, and the gate)."""
    s = fit_apply(Xa, ya, Xb)
    observed = auc_binary(yb, s)
    rng = np.random.default_rng(SEED)
    perms = [rng.permutation(ya) for _ in range(N_PERM)]
    chunks = [perms[i:i + 50] for i in range(0, len(perms), 50)]
    null = np.concatenate(Parallel(n_jobs=N_JOBS)(delayed(_external_chunk)(Xa, c, Xb, yb) for c in chunks))
    return s, observed, (1 + int((null >= observed).sum())) / (1 + N_PERM), float(null.mean())

def boot_gap(ya, Pa, yb, sb):
    """Internal AUC of A (repeat-averaged out of fold) minus the AUC of the A-model on B; A and B resampled independently."""
    rng = np.random.default_rng(SEED)
    vals = []
    while len(vals) < N_BOOT:
        ia = rng.choice(len(ya), len(ya), replace=True)
        ib = rng.choice(len(yb), len(yb), replace=True)
        if len(np.unique(ya[ia])) < 2 or len(np.unique(yb[ib])) < 2:
            continue
        vals.append(np.mean([auc(ya[ia], Pa[r][ia]) for r in range(Pa.shape[0])]) - auc_binary(yb[ib], sb[ib]))
    return mean_auc(ya, Pa) - auc_binary(yb, sb), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

def reading(lo, hi, gap_lo, gap_hi):
    """Plan 7g, experiment B, with the edge cases of settled item 15 (both fixed before any data)."""
    if lo <= 0.5:
        return 'does not transfer'
    if gap_lo > 0:
        return 'transfers with a loss'
    return 'transfers'

def internal(X, y):
    """Notebook 05's probe inside one dataset: 5 x 5 CV, bootstrap CI, permutation test on a fixed split."""
    P, _ = cv_oof_matrix(X, y, repeated_splits(y))
    lo, hi = boot_ci(y, P)
    _, p, null = permutation_test(scaled_folds(X), y)
    return P, {'auc': mean_auc(y, P), 'ci_low': lo, 'ci_high': hi, 'perm_p': p, 'null_mean_auc': float(null.mean())}

def transfers(lo):
    """Notebook 08's rule (7g, settled item 15): a model transfers if its external CI lies wholly above 0.5."""
    return bool(lo > 0.5)


In [ ]:
# 12b. The analyses of plan 7o (G1 tumour and lungs only; G2 habitus; G3 harmonized)
from sklearn.linear_model import LinearRegression
complete = {k for k in REC.key[REC.embedded.fillna(False).astype(bool)]
            if bool(record12(k).get('body_reproduced', False)) and all(has_variant(k, v) for v in INPUTS_B)}

def load12(keys, variant):
    out = {}
    for e in ENCODERS:
        rows = []
        for k in keys:
            with np.load(emb12_path(k)) as z:
                n = f'{e}__{variant}'
                rows.append(z[n] if n in z.files else None)
        d = next((len(a) for a in rows if a is not None), 1)
        out[e] = np.vstack([a if a is not None else np.full(d, np.nan, np.float32) for a in rows]).astype(float)
    return out

def frame12(name, shuffle_seed):
    """Patients of a dataset with all inputs, labels (class 1: squamous; for LIDC-IDRI the second class name)."""
    if name == LIDC_TASK:
        df, y, classes = task_frame(name, shuffle_seed)
        keep = df.key.isin(complete).to_numpy()
        return df[keep].reset_index(drop=True), y[keep], tuple(classes)
    df, y, _ = dataset_frame(name, shuffle_seed)
    return df, y, CLASSES[name]

def strict_p(yt, s):
    """R7: the source model held fixed, the target labels permuted (one-sided, Monte Carlo)."""
    obs = auc_binary(yt, s)
    rng = np.random.default_rng(SEED)
    r = rankdata(s); n1 = int(yt.sum()); n0 = len(yt) - n1
    cnt = 0
    for i in range(0, N_PERM_STRICT, 10000):
        m = min(10000, N_PERM_STRICT - i)
        P = np.argsort(rng.random((m, len(yt))), axis=1) < n1          # random positive sets of the same size
        a = ((P * r).sum(axis=1) - n1 * (n1 + 1) / 2) / (n1 * n0)
        cnt += int((a >= obs - 1e-12).sum())
    return (1 + cnt) / (1 + N_PERM_STRICT)

def holm(p):
    p = np.asarray(p, float); o = np.argsort(p); m = len(p); out = np.empty(m); run = 0.0
    for rank, j in enumerate(o):
        run = max(run, min(1.0, (m - rank) * p[j])); out[j] = run
    return out

def transfer_rows(Xs, ys, frames, X_of, label, enc):
    rows = []
    for t in LUNG_TARGETS:
        dft, yt, _ = frames[t]
        Xt = X_of(t)
        ok = ~np.isnan(Xt).any(axis=1)
        s = fit_apply(Xs, ys, Xt[ok])
        a = auc_binary(yt[ok], s); lo, hi = boot_ci(yt[ok], as_P(s))
        rows.append({'input': label, 'encoder': enc, 'target': t, 'n_target': int(ok.sum()),
                     'n_target_left_out': int((~ok).sum()), 'external_auc': a, 'ci_low': lo, 'ci_high': hi,
                     'transfers_ci': bool(lo > 0.5), 'strict_p': strict_p(yt[ok], s)})
    return rows

def internal_row(name, label, enc, X, y):
    ok = ~np.isnan(X).any(axis=1)
    if ok.sum() < 20 or len(np.unique(y[ok])) < 2 or min(np.bincount(y[ok])) < 5:
        return {'dataset': name, 'input': label, 'encoder': enc, 'n': int(ok.sum()), 'auc': np.nan}
    _, r = internal(X[ok], y[ok])
    return {'dataset': name, 'input': label, 'encoder': enc, 'n': int(ok.sum()), 'n_left_out': int((~ok).sum()),
            'n_class1': int(y[ok].sum()), **r}

def residual_probe_oof(X, H, y, splits):
    """Background embeddings with the habitus measures regressed out inside each training fold (plan 7o, G2c)."""
    k = 2; P = np.zeros((len(splits), len(y), k))
    for r, folds in enumerate(splits):
        for tr, te in folds:
            hp = Pipeline([('fill', SimpleImputer(strategy='median')), ('scale', StandardScaler())]).fit(H[tr])
            reg = LinearRegression().fit(hp.transform(H[tr]), X[tr])
            Rtr = X[tr] - reg.predict(hp.transform(H[tr])); Rte = X[te] - reg.predict(hp.transform(H[te]))
            m = Pipeline([('scale', StandardScaler()), ('model', LogisticRegression(**LR_PARAMS))]).fit(Rtr, y[tr])
            P[r, te] = m.predict_proba(Rte)
    return P

def residual_fit_apply(Xs, Hs, ys, Xt, Ht):
    hp = Pipeline([('fill', SimpleImputer(strategy='median')), ('scale', StandardScaler())]).fit(Hs)
    reg = LinearRegression().fit(hp.transform(Hs), Xs)
    return fit_apply(Xs - reg.predict(hp.transform(Hs)), ys, Xt - reg.predict(hp.transform(Ht)))

def habitus_of(df):
    return pd.DataFrame([record12(k) for k in df.key]).reindex(columns=HABITUS).astype(float).to_numpy()

def read_transfer(rows):
    """Plan 7o readings, per input and encoder, after Holm over all comparisons of that experiment."""
    t = pd.DataFrame(rows)
    t['strict_p_holm'] = holm(t.strict_p)
    t['transfers'] = t.transfers_ci & (t.strict_p_holm < 0.05)
    out = []
    for (inp, enc), g in t.groupby(['input', 'encoder']):
        ind = int(g[g.target.isin(INDEPENDENT)].transfers.sum())
        cp = bool(g[g.target == 'CPTAC lung'].transfers.any())
        reading = ('generalizes' if ind >= 2 else 'behaves like the background' if (cp and ind == 0)
                   else 'no transferable signal' if (ind == 0 and not cp) else 'mixed')
        out.append({'input': inp, 'encoder': enc, 'independent_cohorts_transferred': ind, 'cptac_transferred': cp,
                    'reading': reading})
    return t, pd.DataFrame(out)

def analyse12(mode, shuffle_seed):
    frames = {n: frame12(n, shuffle_seed) for n in DATASETS12}
    res = {'g1_internal': [], 'g1_transfer': [], 'g2_by_class': [], 'g2_internal': [], 'g2_transfer': [],
           'g3_internal': [], 'g3_transfer': [], 'detected_by_class': [], 'labels': []}
    for name, (df, y, cls) in frames.items():          # for notebook 13 (the 3D network reads them)
        res['labels'] += [{'dataset': name, 'key': k, 'y': int(v), 'class': cls[int(v)]} for k, v in zip(df.key, y)]
    # G1: tumour only and lungs only
    for name, (df, y, cls) in frames.items():
        det = np.array([bool(record12(k).get('tumour_detected')) for k in df.key])
        for j, c in enumerate(cls):
            res['detected_by_class'].append({'dataset': name, 'class': c, 'patients': int((y == j).sum()),
                                             'tumour_detected': int(det[y == j].sum())})
        for v in INPUTS_G1:
            E = load12(df.key, v)
            for enc in ENCODERS:
                res['g1_internal'].append(internal_row(name, v, enc, E[enc], y))
    dfs, ys, _ = frames[SOURCE]
    for v in INPUTS_G1:
        Es = load12(dfs.key, v)
        Et = {t: load12(frames[t][0].key, v) for t in LUNG_TARGETS}
        for enc in ENCODERS:
            ok = ~np.isnan(Es[enc]).any(axis=1)
            res['g1_transfer'] += transfer_rows(Es[enc][ok], ys[ok], frames, lambda t: Et[t][enc], v, enc)
    # G2: habitus by class, habitus model, residualized background
    for name in (SOURCE,) + LUNG_TARGETS:
        df, y, _ = frames[name]
        H = habitus_of(df)
        for i, m in enumerate(HABITUS):
            x = H[:, i]; ok = np.isfinite(x)
            a = auc_binary(y[ok], x[ok]) if len(np.unique(y[ok])) == 2 else np.nan
            lo, hi = boot_ci(y[ok], as_P(x[ok])) if np.isfinite(a) else (np.nan, np.nan)
            res['g2_by_class'].append({'dataset': name, 'measure': m, 'n': int(ok.sum()),
                                       'auc_squamous_higher': a, 'ci_low': lo, 'ci_high': hi})
    Hs = habitus_of(dfs)
    hmodel = lambda: Pipeline([('fill', SimpleImputer(strategy='median')), ('scale', StandardScaler()),
                               ('model', LogisticRegression(**LR_PARAMS))])
    Ph = np.zeros((N_REPEATS, len(ys), 2))
    for r, folds in enumerate(repeated_splits(ys)):
        for tr, te in folds:
            Ph[r, te] = hmodel().fit(Hs[tr], ys[tr]).predict_proba(Hs[te])
    lo, hi = boot_ci(ys, Ph)
    res['g2_internal'].append({'model': 'habitus', 'encoder': '-', 'auc': mean_auc(ys, Ph), 'ci_low': lo, 'ci_high': hi})
    hm = hmodel().fit(Hs, ys)
    Ebg_s = load_inputs(dfs.key)
    for t in LUNG_TARGETS:
        dft, yt, _ = frames[t]
        s = hm.decision_function(habitus_of(dft))
        a = auc_binary(yt, s); l2, h2 = boot_ci(yt, as_P(s))
        res['g2_transfer'].append({'input': 'habitus model', 'encoder': '-', 'target': t, 'n_target': len(yt),
                                   'external_auc': a, 'ci_low': l2, 'ci_high': h2, 'transfers_ci': bool(l2 > 0.5),
                                   'strict_p': strict_p(yt, s)})
    for enc in ENCODERS:
        Xs = Ebg_s[(enc, 'background')]
        P = residual_probe_oof(Xs, Hs, ys, repeated_splits(ys)); l2, h2 = boot_ci(ys, P)
        res['g2_internal'].append({'model': 'background, habitus removed', 'encoder': enc, 'auc': mean_auc(ys, P),
                                   'ci_low': l2, 'ci_high': h2})
        for t in LUNG_TARGETS:
            dft, yt, _ = frames[t]
            Xt = load_inputs(dft.key)[(enc, 'background')]
            s = residual_fit_apply(Xs, Hs, ys, Xt, habitus_of(dft))
            a = auc_binary(yt, s); l3, h3 = boot_ci(yt, as_P(s))
            res['g2_transfer'].append({'input': 'background, habitus removed', 'encoder': enc, 'target': t,
                                       'n_target': len(yt), 'external_auc': a, 'ci_low': l3, 'ci_high': h3,
                                       'transfers_ci': bool(l3 > 0.5), 'strict_p': strict_p(yt, s)})
    # G3: harmonized geometry
    for v in INPUTS_G3:
        Es = load12(dfs.key, v)
        Et = {t: load12(frames[t][0].key, v) for t in LUNG_TARGETS}
        for enc in ENCODERS:
            res['g3_internal'].append(internal_row(SOURCE, v, enc, Es[enc], ys))
            res['g3_transfer'] += transfer_rows(Es[enc], ys, frames, lambda t: Et[t][enc], v, enc)
    return res

def readings12(res):
    g1t, g1r = read_transfer(res['g1_transfer'])
    if not TUMOUR_CHECK.get('segmentation_reliable', False):
        g1r.loc[g1r.input == 'tumour', 'reading'] = 'unreliable segmentation (not interpreted)'
    g2t = pd.DataFrame(res['g2_transfer']); g2t['strict_p_holm'] = holm(g2t.strict_p)
    g2t['transfers'] = g2t.transfers_ci & (g2t.strict_p_holm < 0.05)
    hab_cp = bool(g2t[(g2t.input == 'habitus model') & (g2t.target == 'CPTAC lung')].transfers.any())
    res_cp = {e: bool(g2t[(g2t.input == 'background, habitus removed') & (g2t.encoder == e) &
                          (g2t.target == 'CPTAC lung')].transfers.any()) for e in ENCODERS}
    g2r = pd.DataFrame([{'encoder': e, 'habitus_model_transfers_to_cptac': hab_cp,
                         'residualized_background_transfers_to_cptac': res_cp[e],
                         'reading': ('accounts for the CPTAC transfer' if hab_cp and not res_cp[e] else
                                     'does not account for it' if res_cp[e] else 'partly')} for e in ENCODERS])
    g3t, _ = read_transfer(res['g3_transfer'])
    g3i = pd.DataFrame(res['g3_internal'])
    g3r = []
    for e in ENCODERS:
        bi = g3i[(g3i.input == 'harm_background') & (g3i.encoder == e)].iloc[0]
        cp = bool(g3t[(g3t.input == 'harm_background') & (g3t.encoder == e) & (g3t.target == 'CPTAC lung')].transfers.any())
        g3r.append({'encoder': e, 'internal_ci_includes_0.5': bool(bi.ci_low <= 0.5), 'cptac_transferred': cp,
                    'reading': ('removes the tissue-free signal' if bi.ci_low <= 0.5 and not cp else
                                'does not remove it' if cp else 'reduces it')})
    return {'g1_transfer': g1t, 'g1_reading': g1r, 'g2_transfer': g2t, 'g2_reading': g2r, 'g3_transfer': g3t,
            'g3_reading': pd.DataFrame(g3r)}

In [ ]:
# 13b. Run one mode (rehearsal on shuffled labels, or real), with saving and a run record
def run_mode12(mode):
    out = OUT_ROOT / mode
    out.mkdir(parents=True, exist_ok=True)
    shuffle_seed = 999 if mode == 'rehearsal' else (1000 if TEST.get('real_is_shuffled') else None)
    print(f'\n===== {mode.upper()} pass ({"shuffled labels" if shuffle_seed is not None else "REAL labels"}) =====')
    t0 = time.time()
    res = analyse12(mode, shuffle_seed)
    rd = readings12(res)
    tables = {'g1_internal.csv': pd.DataFrame(res['g1_internal']), 'g1_transfer.csv': rd['g1_transfer'],
              'g1_reading.csv': rd['g1_reading'], 'tumour_detected_by_class.csv': pd.DataFrame(res['detected_by_class']),
              'g2_habitus_by_class.csv': pd.DataFrame(res['g2_by_class']), 'g2_internal.csv': pd.DataFrame(res['g2_internal']),
              'g2_transfer.csv': rd['g2_transfer'], 'g2_reading.csv': rd['g2_reading'],
              'g3_internal.csv': pd.DataFrame(res['g3_internal']), 'g3_transfer.csv': rd['g3_transfer'],
              'g3_reading.csv': rd['g3_reading'], 'dataset_labels.csv': pd.DataFrame(res['labels'])}
    for n, t in tables.items():
        t.to_csv(out / n, index=False)
    print(f'Analyses finished in {(time.time() - t0) / 60:.1f} min')
    for n in ('g1_reading.csv', 'g2_reading.csv', 'g3_reading.csv'):
        show(tables[n])
    outputs = [{'file': p.name, 'bytes': p.stat().st_size, 'md5': md5(p)} for p in sorted(out.glob('*.csv'))]
    manifest = {'notebook': '12_image_experiments.ipynb', 'mode': mode, 'plan': '7o (G1-G3)',
                'labels': 'real' if shuffle_seed is None else 'shuffled (rehearsal)' if mode == 'rehearsal' else 'shuffled (local test)',
                'run_started_utc': RUN_STARTED,
                'run_finished_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(timespec='seconds'),
                'in_colab': IN_COLAB, 'local_test_settings': TEST, 'input_runs': runs, 'input_caches': caches,
                'totalsegmentator_version': TS_VERSION, 'totalsegmentator_torch': TS_TORCH, 'totalsegmentator_nnunetv2': TS_NNUNET, 'totalsegmentator_environment': TS_CHECK, 'tumour_check': TUMOUR_CHECK, 'coverage': cov12.to_dict('records'),
                'settings': {'PREP12_KEY': PREP12_KEY, 'EMB12_KEY': EMB12_KEY, 'LUNG_DILATE_MM': LUNG_DILATE_MM,
                             'TUMOUR_DILATE_MM': TUMOUR_DILATE_MM, 'HABITUS': HABITUS, 'N_BOOT': N_BOOT, 'N_PERM': N_PERM,
                             'N_PERM_STRICT': N_PERM_STRICT, 'SEED': SEED, 'LR_PARAMS': LR_PARAMS},
                'versions': VERSIONS, 'outputs': outputs}
    (out / 'run_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
    return tables

modes = {'both': ['rehearsal', 'real'], 'rehearsal': ['rehearsal'], 'real': ['real']}[RUN_MODE]
results12 = {}
for mode in modes:
    if mode == 'real' and 'rehearsal' in results12:
        t = results12['rehearsal']
        nulls = pd.concat([t['g1_internal.csv'].null_mean_auc.dropna(), t['g3_internal.csv'].null_mean_auc.dropna()])
        problems = []
        if not nulls.between(*GATE_NULL_RANGE).all():
            problems.append(f'a null mean AUC is outside {GATE_NULL_RANGE}: {nulls.round(3).tolist()}')
        if (cov12.coverage < GATE_MIN_COVERAGE).any():
            problems.append('too few scans finished for a dataset')
        if problems:
            print('\nGATE NOT PASSED; the real-label pass was NOT run:', *problems, sep='\n - ')
            break
        print('\nRehearsal gate passed: null means look like chance and enough scans are finished.')
    results12[mode] = run_mode12(mode)

In [ ]:
# 14b. Plain-English summary, read back from the saved files
for mode in results12:
    out = OUT_ROOT / mode
    lab_kind = json.loads((out / 'run_manifest.json').read_text())['labels']
    print(f"\n{mode.upper()} ({lab_kind})")
    print('Tumour segmentation check:', TUMOUR_CHECK)
    for f in ('g1_reading.csv', 'g2_reading.csv', 'g3_reading.csv'):
        for r in pd.read_csv(out / f).to_dict('records'):
            print(f'- {f[:2].upper()}:', {k: v for k, v in r.items()})
print('\nNext: review the saved results and run log before writing up findings.')

In [ ]:
# 15. Stop the runtime once everything above has finished (notebook 08, unchanged)
# "Run all" stops at the first error, so if any cell fails, this cell never runs and the runtime stays on for checking.
if IN_COLAB:
    from google.colab import runtime
    print('Notebook finished:', time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime()))
    drive.flush_and_unmount()   # push every result file to Drive before the machine is released
    print('Drive synced. Stopping the runtime now.')
    time.sleep(5)
    runtime.unassign()